# Faster R-CNN · 2차 데이터 실험

**v2·v3·v4 × 독립 5·10·20 epoch · seed 0 · 실효 batch 16(실제 16 × 누적 1, 16GB GPU는 8 × 2) · Colab → Kaggle 재개**

이 노트북은 기존 02 노트북을 수정한 것입니다. 1차 원본은 `exp/23-faster-rcnn-baseline` 브랜치에 있습니다. 팀 `main`의 2차 데이터·평가 코드를 사용하며, 공통 코드와 고정 CSV를 수정하지 않습니다.

실행 순서: **v2e5 → v3e5 → v4e5 → v2e10 → v3e10 → v4e10 → v2e20 → v3e20 → v4e20**. 각 run은 COCO_V1에서 새로 시작합니다. 5 epoch 가중치를 10 epoch 초기값으로 쓰지 않습니다. Warmup 3 epoch·cosine은 총 예산에 따라 달라집니다.

| 고정 조건 | 값 |
| --- | --- |
| 모델 / 학습 범위 | Faster R-CNN ResNet50 FPN / 전체 FT |
| Optimizer / LR | AdamW / 3e-4, weight decay 1e-4 |
| 실제 batch / 누적 | 16 / 1, 16GB GPU(T4·P100)는 8 / 2 (실효 batch 16 고정, 1번 셀 `MICRO_BATCH`) |
| 학습 / 검증 정밀도 | AMP FP16 / FP32 |
| 입력 / 정규화 | min=max 640 / FrozenBatchNorm eps=0 고정 |
| 증강 / seed | 없음 / 0 |
| 속도 설정 | 결정적 연산 끔(cudnn.benchmark), 로더 worker는 CPU 수에 맞춤(최대 8), val loss·추론 batch 8 |
| Best 선택 | 전체 공식 `val` mAP75:95 최대 |
| 진단 | score 0.30·0.50, IoU 0.75 |
| 공식 AP | score 하한 0.001, 팀 무시 규칙 |

**처음에는 수정 검토·커밋·push부터 함께 진행합니다.** 아래 작업 스위치는 모두 꺼져 있습니다. 환경·데이터 준비, 짧은 전환 검사, 본 학습을 각각 확인하고 실행합니다. 모든 셀 실행으로 장시간 학습이 시작되지 않습니다. 이 노트북은 클라우드 검사를 통과했다고 가정하지 않습니다.

- [데이터 공유 폴더](https://drive.google.com/drive/folders/1mUFpgZaGB7s2ilIWF1Dl4KtPsUWsFp8K): `pill data/data`의 ZIP 4개
- [결과 공유 폴더](https://drive.google.com/drive/folders/1ONSH_SlukXeXGkh6Z1ddfq0XawBJv-9c): `pill data/cj_runs`
- [W&B 팀 프로젝트](https://wandb.ai/codeit-prj1-team2/pill-object-detection)

결과 공유: **11월 2일(월) 18:00 KST**. 첫 run의 실제 시간을 보고 남은 예산을 판단합니다. 9개 중 완료·중단·미실행을 비교표에 구분합니다.

## 1. 실행 환경과 경로

Colab에 공유 `pill data` 바로가기를 MyDrive에 추가한 뒤 실제 경로를 확인합니다. Kaggle은 Internet을 켜고 GPU를 선택합니다. Python과 torch 등은 Colab에 원래 있는 것을 쓰고, 없는 `pycocotools`, `wandb`만 설치합니다.

`BRANCH`는 현재 로컬 브랜치입니다. **변경한 노트북을 이 브랜치에 push한 뒤** 클라우드에서 사용하세요. 기존 클라우드 저장소는 자동으로 pull·reset·브랜치 전환하지 않습니다. 코드가 오래됐다면 변경 상태를 확인하고 직접 최신화합니다.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import shutil
import json
import zipfile
import stat
import tempfile
import tomllib

PLATFORM = "colab" if "google.colab" in sys.modules or Path("/content").exists() else (
    "kaggle" if Path("/kaggle").exists() else "local")

REPOSITORY_URL = "https://github.com/codeit-ai-15th/pill-object-detection.git"

BRANCH = "exp/rw-faster-rcnn-exp2"  # 이 노트북을 커밋·push한 뒤 클라우드에서 사용

REPO_ROOT = Path("/content/pill-object-detection" if PLATFORM == "colab" else (
    "/kaggle/working/pill-object-detection" if PLATFORM == "kaggle" else
    "/Users/codeit/Desktop/codeit/Project1/pill-object-detection"))

DATA_DRIVE_FOLDER_ID = "1mUFpgZaGB7s2ilIWF1Dl4KtPsUWsFp8K" 

RESULT_DRIVE_FOLDER_ID = "11zbJt06epCnfDSLY-UI-BMkFBv4KJmjB"

DRIVE_ROOT = "/content/drive/MyDrive/CodeIT/Colab/project1-pill-object-detection/pill data"  # Colab: zip(data/)과 결과(rw_runs/)를 두는 Drive 폴더

ZIP_ROOT = Path(f"{DRIVE_ROOT}/data" if PLATFORM == "colab" else (
    "/kaggle/working/team_zips" if PLATFORM == "kaggle" else "/private/tmp/cj-team-zips"))

RUN_ROOT = Path(f"{DRIVE_ROOT}/rw_runs" if PLATFORM == "colab" else (
    "/kaggle/working/cj_runs" if PLATFORM == "kaggle" else REPO_ROOT / "notebooks/modeling/02-cj-faster-rcnn-exp2"))

DATASET_ROOT = Path("/content/exp2" if PLATFORM == "colab" else (
    "/kaggle/working/exp2" if PLATFORM == "kaggle" else REPO_ROOT / "data/processed/exp2"))

VARIANTS = ("v3",)  # 돌릴 방법. 하나만 넣을 때도 쉼표를 붙인다. 예: ("v4",)

EPOCH_BUDGETS = (5, 10, 20)

RESUME_RUN_NAMES = set()  # 예: {"v3_frcnn_e10_s0"}; 동일 run의 총 epoch는 바꾸지 않는다.

MICRO_BATCH = 16  # 실제 batch. 실효 batch 16은 고정이고 누적은 16 ÷ 이 값. 16GB GPU(T4·P100)에서 메모리가 부족하면 8

print("환경:", PLATFORM, "| 저장소:", REPO_ROOT)
print("결과 위치:", RUN_ROOT)


## 2. 환경 준비와 확인

클라우드에서는 레포를 받고(이미 있으면 건너뜀) Colab에 없는 `pycocotools`, `wandb`만 설치합니다. 팀 Colab 가이드(EXPERIMENTS.md)와 같은 버전입니다. torch, numpy 등 이미 있는 패키지는 바꾸지 않으므로 커널을 다시 시작하지 않아도 됩니다. 실제로 쓴 버전은 각 run의 `environment_history.json`과 W&B config에 기록됩니다.

In [ ]:
# 클라우드에서는 레포를 받고, Colab에 없는 패키지만 설치한다. torch, numpy 등은 Colab에 있는 것을 그대로 쓴다.
# 이미 불러온 패키지를 바꾸지 않으므로 커널을 다시 시작할 필요가 없다.
if PLATFORM != "local":
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY_URL, str(REPO_ROOT)], check=True)
    branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO_ROOT, text=True).strip()
    if branch != BRANCH:
        raise RuntimeError(f"다른 브랜치입니다: {branch}. 작업 파일을 확인한 뒤 직접 전환하세요.")
    # 팀 Colab 가이드(EXPERIMENTS.md)와 같은 버전
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools==2.0.11", "wandb==0.31.0"],
                            capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f"패키지 설치 실패:\n{result.stderr}")
    print("레포·패키지 준비 완료:", REPO_ROOT)
else:
    print("로컬은 uv sync로 준비한 환경을 그대로 씁니다.")


In [ ]:
if not REPO_ROOT.is_dir():
    raise FileNotFoundError("저장소가 없습니다. 변경한 노트북을 push한 뒤 위 준비 셀을 실행하세요.")
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO_ROOT, text=True).strip()
if branch != BRANCH:
    raise RuntimeError(f"브랜치가 다릅니다: {branch}. 파일 상태를 확인한 뒤 직접 전환하세요.")
for name in ("EXPERIMENTS.md", "scripts/build_dataset.py", "src/pill_detection/evaluate.py"):
    if not (REPO_ROOT / name).is_file():
        raise FileNotFoundError(f"팀 2차 실험 파일이 없습니다: {name}")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.environ["WANDB_ENTITY"] = "codeit-prj1-team2"
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import importlib.metadata
import torch
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda)
print("패키지:", {name: importlib.metadata.version(name) for name in
                ("torchvision", "numpy", "pandas", "pillow", "pycocotools", "wandb")})
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음(로컬 계산 검사만 가능)")
if PLATFORM != "local" and not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU가 없습니다. 런타임 유형을 GPU로 바꾸세요.")


## 3. 공통 함수 정의

이 구역은 함수를 정의합니다. 학습·추론·Drive 전송을 시작하지 않습니다. 각 함수 셀의 실제 소스와 팀 공통 코드 해시를 저장해 다른 코드로 재개하는 일을 막습니다. Colab·Kaggle의 경로와 GPU는 달라도 되며, 데이터·설정·코드·핵심 패키지는 같아야 합니다.

In [ ]:
"""2차 실험의 공통 도구와 실제 실행 코드 식별자."""
import ast
import contextlib
import copy
import csv
import gc
import hashlib
import importlib.metadata
import inspect
import io
import json
import linecache
import math
import os
import random
import shutil
import subprocess
import sys
import tempfile
import time
import uuid
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights, fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import FrozenBatchNorm2d, box_convert, box_iou
from pill_detection.utils import xyxy_to_xywh, draw_boxes
from pill_detection.datasets import OTHERS_CATEGORY_ID
from pill_detection.evaluate import (
    evaluate_val, drop_ignored, load_gt, competition_from_classes, write_submission,
)

NOTEBOOK_CODE_BLOCKS = {}
CORE_PACKAGES = ("torch", "torchvision", "numpy", "pandas", "pillow", "pycocotools", "wandb")


def _register_notebook_block(name):
    """실행한 함수 셀의 소스를 기록한다."""
    frame = inspect.currentframe().f_back
    try:
        source = "".join(linecache.getlines(frame.f_code.co_filename))
        if not source:
            source = get_ipython().history_manager.input_hist_raw[-1]
        if not source.strip():
            raise RuntimeError("실행 셀의 소스를 찾지 못했습니다. 노트북 함수 셀을 위에서부터 다시 실행하세요.")
        NOTEBOOK_CODE_BLOCKS[name] = source
    finally:
        del frame


def file_sha256(path):
    """파일 내용의 SHA-256을 반환한다."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def stable_hash(value):
    """경로나 셀 실행 번호와 무관한 JSON 식별자를 만든다."""
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     allow_nan=False).encode()).hexdigest()


def write_json(path, value):
    """완성한 JSON만 교체해 중단 중인 파일을 결과로 사용하지 않는다."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent,
                                     prefix=path.name + ".", suffix=".part", delete=False) as stream:
        temporary = Path(stream.name)
        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)
    os.replace(temporary, path)


def write_frame(path, frame):
    """CSV를 임시 파일에서 완성한 뒤 교체한다."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".part")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def code_identity():
    """실제로 실행한 함수 셀과 팀 공통 코드의 해시를 반환한다."""
    names = ("helpers", "data", "storage", "analysis", "training")
    missing = set(names) - set(NOTEBOOK_CODE_BLOCKS)
    if missing:
        raise RuntimeError(f"함수 셀이 빠졌습니다: {sorted(missing)}. 함수 셀을 위에서부터 실행하세요.")
    embedded = {name: stable_hash(ast.dump(ast.parse(NOTEBOOK_CODE_BLOCKS[name]),
                                         include_attributes=False)) for name in names}
    common = {name: file_sha256(REPO_ROOT / "src/pill_detection" / name)
              for name in ("utils.py", "data.py", "datasets.py", "evaluate.py")}
    return {"notebook": embedded, "common": common}


def package_versions():
    """재개 비교용 기본 버전을 반환하고 CUDA 빌드 접미사는 별도로 기록한다."""
    return {name: importlib.metadata.version(name).split("+")[0] for name in CORE_PACKAGES}


def runtime_info():
    """실행 환경은 재개 식별자와 구분해 이력으로 기록한다."""
    result = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT,
                            capture_output=True, text=True, check=True)
    dirty = subprocess.run(["git", "status", "--porcelain"], cwd=REPO_ROOT,
                           capture_output=True, text=True, check=True)
    return {"time_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "platform": PLATFORM, "python": sys.version.split()[0],
            "python_executable": sys.executable, "packages": {
                name: importlib.metadata.version(name) for name in CORE_PACKAGES},
            "cuda": torch.version.cuda,
            "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            "repository_commit": result.stdout.strip(), "repository_dirty": bool(dirty.stdout.strip()),
            "lock_sha256": file_sha256(REPO_ROOT / "uv.lock")}


def save_figure(path, figure):
    """그림을 저장하고 노트북에 표시한 뒤 메모리를 반환한다."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(path, dpi=150, bbox_inches="tight")
    display(figure)
    plt.close(figure)


_register_notebook_block("helpers")


In [ ]:
"""팀에서 생성한 COCO 데이터와 원래 K-코드 평가 정답을 연결한다."""
IMAGE_HASH_CACHE = {}
EXPECTED_DATA = {"v2": (7129, 118), "v3": (8974, 96), "v4": (11206, 75)}
BUNDLES = {}


def image_digest(path):
    """같은 세션에서 동일한 이미지 내용을 여러 방법이 다시 읽지 않도록 캐시한다."""
    path = Path(path)
    stat = path.stat()
    key = (str(path.resolve()), stat.st_size, stat.st_mtime_ns)
    if key not in IMAGE_HASH_CACHE:
        IMAGE_HASH_CACHE[key] = file_sha256(path)
    return IMAGE_HASH_CACHE[key]


class Exp2Dataset(Dataset):
    """COCO categories의 idx+1을 배경 0과 분리해 torchvision에 전달한다."""

    def __init__(self, image_dir, annotation_path, label_of):
        self.image_dir = Path(image_dir)
        self.doc = json.loads(Path(annotation_path).read_text(encoding="utf-8"))
        self.frame = pd.DataFrame(self.doc["images"]).rename(columns={"id": "image_id"})
        self.frame = self.frame.sort_values("image_id").reset_index(drop=True)
        if self.frame["file_name"].duplicated().any() or self.frame["image_id"].duplicated().any():
            raise ValueError("이미지 이름·ID가 중복됩니다. 팀 학습 폴더를 다시 확인하세요.")
        self.annotations = defaultdict(list)
        self.label_of = label_of
        image_ids = set(self.frame["image_id"])
        for ann in self.doc["annotations"]:
            if ann["image_id"] not in image_ids or ann["category_id"] not in label_of:
                raise ValueError("COCO annotation의 이미지·클래스 연결이 다릅니다. 학습 폴더를 확인하세요.")
            self.annotations[ann["image_id"]].append(ann)
        for row in self.frame.itertuples():
            if Path(row.file_name).name != row.file_name or not (self.image_dir / row.file_name).is_file():
                raise FileNotFoundError(f"학습 이미지가 없습니다: {row.file_name}. ZIP 전개와 데이터 경로를 확인하세요.")

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(self.image_dir / row.file_name) as image:
            if image.size != (int(row.width), int(row.height)):
                raise ValueError(f"이미지 크기가 COCO 정답과 다릅니다: {row.file_name}. 데이터를 확인하세요.")
            tensor = pil_to_uint8(image)
        anns = self.annotations[int(row.image_id)]
        boxes = torch.tensor([ann["bbox"] for ann in anns], dtype=torch.float32).reshape(-1, 4)
        if boxes.numel() and (not torch.isfinite(boxes).all() or (boxes[:, 2:] <= 0).any()):
            raise ValueError(f"유효하지 않은 xywh bbox입니다: {row.file_name}. 학습 폴더를 확인하세요.")
        areas = boxes[:, 2] * boxes[:, 3]
        return tensor, {"boxes": box_convert(boxes, "xywh", "xyxy"),
                        "labels": torch.tensor([self.label_of[a["category_id"]] for a in anns], dtype=torch.int64),
                        "image_id": torch.tensor(int(row.image_id)), "area": areas,
                        "iscrowd": torch.tensor([a.get("iscrowd", 0) for a in anns], dtype=torch.int64)}


class TestImages(Dataset):
    """정답 없는 test 이미지도 누락 없이 manifest에 남긴다."""

    def __init__(self, image_dir):
        self.image_dir = Path(image_dir)
        rows = []
        for path in sorted(self.image_dir.glob("*.png"), key=lambda p: p.name):
            if not path.stem.isdigit():
                raise ValueError("Test 파일명은 숫자.png여야 합니다. 대회 test 폴더를 확인하세요.")
            with Image.open(path) as image:
                rows.append({"file_name": path.name, "image_id": int(path.stem),
                             "width": image.width, "height": image.height})
        self.frame = pd.DataFrame(rows)
        if self.frame.empty or self.frame["image_id"].duplicated().any():
            raise ValueError("Test 이미지가 없거나 ID가 중복됩니다. test_images.zip과 경로를 확인하세요.")
        self.frame = self.frame.sort_values("image_id").reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(self.image_dir / row.file_name) as image:
            tensor = pil_to_uint8(image)
        return tensor, {"image_id": torch.tensor(int(row.image_id))}


def collate_detection(batch):
    """이미지와 객체 수가 달라도 작은 배치로 묶는다."""
    return tuple(zip(*batch))


def pil_to_uint8(image):
    """PIL 이미지를 uint8 CHW tensor로 바꾼다. float32보다 4배 작아 worker 전달·GPU 복사가 빠르다."""
    return torch.from_numpy(np.array(image.convert("RGB"), dtype=np.uint8)).permute(2, 0, 1).contiguous()


def to_model_inputs(images, device):
    """uint8 이미지를 GPU로 옮긴 뒤 to_tensor와 같은 0~1 float32로 바꾼다."""
    return [image.to(device, non_blocking=True).to(torch.float32).div(255) for image in images]


def prepare_bundle(variant, expected=True):
    """방법별 학습 데이터와 변하지 않는 공식 평가 정답을 한 번 준비한다."""
    if variant in BUNDLES:
        return BUNDLES[variant]
    root = DATASET_ROOT / variant
    required = [root / name for name in ("coco/train.json", "coco/val.json", "eval/val_gt.json",
                                         "classes.csv", "dataset_info.json")]
    if any(not p.is_file() for p in required):
        raise FileNotFoundError(f"{variant} 학습 폴더가 없습니다. 데이터 생성 셀을 먼저 실행하세요.")
    info = json.loads((root / "dataset_info.json").read_text())
    if info.get("variant") != variant:
        raise ValueError("dataset_info의 방법이 다릅니다. 학습 폴더를 확인하세요.")
    if expected:
        for name, key in (("images.csv", "images_csv_sha256"), ("classes.csv", "classes_csv_sha256")):
            if info.get(key) != file_sha256(REPO_ROOT / "data/datasets" / name):
                raise ValueError("학습 폴더가 현재 팀 목록과 다릅니다. 데이터를 다시 준비하세요.")
    cats = json.loads((root / "coco/train.json").read_text())["categories"]
    if sorted(c["idx"] for c in cats) != list(range(len(cats))):
        raise ValueError("COCO categories.idx가 연속적이지 않습니다. 팀 생성 데이터를 확인하세요.")
    label_of = {int(c["id"]): int(c["idx"]) + 1 for c in cats}
    if len(label_of) != len(cats):
        raise ValueError("중복 클래스 ID입니다. 팀 생성 데이터를 확인하세요.")
    datasets = {split: Exp2Dataset(root / "images" / split, root / "coco" / f"{split}.json", label_of)
                for split in ("train", "val")}
    if datasets["val"].doc["categories"] != cats:
        raise ValueError("Train·val 클래스 매핑이 다릅니다. 학습 폴더를 확인하세요.")
    gt = load_gt(root / "eval/val_gt.json")
    if set(datasets["train"].frame.file_name) & set(datasets["val"].frame.file_name):
        raise ValueError("Train과 val 이미지가 겹칩니다. 팀 목록을 확인하세요.")
    if set(datasets["val"].frame.file_name) != {image["file_name"] for image in gt["images"]}:
        raise ValueError("학습용 val과 공식 정답 이미지가 다릅니다. 같은 방법의 학습 폴더를 확인하세요.")
    if expected and (len(datasets["train"]), len(cats), len(gt["images"]), len(gt["annotations"])) != (
            *EXPECTED_DATA[variant], 1000, 3790):
        raise ValueError("팀 안내의 데이터 개수와 다릅니다. --check와 데이터 전개 상태를 확인하세요.")
    for other in BUNDLES.values():
        if stable_hash(other["gt"]) != stable_hash(gt):
            raise ValueError("방법 사이의 공식 val 정답이 다릅니다. 팀 데이터 목록을 확인하세요.")
    print(f"{variant}: 이미지 내용 해시 확인 중(세션에서 같은 이미지는 재사용)…", flush=True)
    image_hashes = {split: {r.file_name: image_digest(ds.image_dir / r.file_name)
                             for r in ds.frame.itertuples()} for split, ds in datasets.items()}
    if set(image_hashes["train"].values()) & set(image_hashes["val"].values()):
        raise ValueError("내용이 같은 이미지가 train/val에 겹칩니다. 데이터 담당자에게 확인하세요.")
    fingerprint = {"variant": variant, "files": {p.relative_to(root).as_posix(): file_sha256(p)
                    for p in required if p.name != "dataset_info.json"}, "images": image_hashes}
    mapping = pd.DataFrame([{"category_id": int(c["id"]), "model_label": label_of[c["id"]],
                             "k_code": str(c.get("k_code", c["id"])),
                             "name": str(c.get("product_name", c.get("name", c["id"])))} for c in cats])
    bundle = {"variant": variant, "root": root, "datasets": datasets, "gt": gt,
              "class_map": mapping, "category_to_label": label_of,
              "label_to_category": {v: k for k, v in label_of.items()},
              "identity": stable_hash(fingerprint), "dataset_info": info, "fingerprint": fingerprint}
    BUNDLES[variant] = bundle
    return bundle


def make_dataset(bundle, split):
    """학습과 validation loss에는 방법별 COCO 라벨을 사용한다."""
    return bundle["datasets"][split]


def show_data(bundle):
    """개수·클래스 연결과 실제 정답 bbox를 보여준다."""
    rows = [{"split": split, "images": len(ds), "boxes": len(ds.doc["annotations"])}
            for split, ds in bundle["datasets"].items()]
    display(pd.DataFrame(rows))
    display(bundle["class_map"])
    dataset = make_dataset(bundle, "train")
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for axis, index in zip(axes, range(min(3, len(dataset)))):
        image, target = dataset[index]
        labels = [str(bundle["label_to_category"][int(v)]) for v in target["labels"]]
        picture = draw_boxes(dataset.image_dir / dataset.frame.iloc[index].file_name,
                             [xyxy_to_xywh(b.tolist()) for b in target["boxes"]], labels, color="lime")
        axis.imshow(picture); axis.axis("off")
    plt.show(); plt.close(fig)


def predictions_for_team(records):
    """추론 캐시를 팀 공식 함수의 K-코드·xywh 형식으로 변환한다."""
    return [{"file_name": row["file_name"], "category_id": int(label),
             "bbox": xyxy_to_xywh(box.tolist()), "score": float(score)}
            for row in records for box, label, score in zip(
                row["prediction"]["boxes"], row["prediction"]["labels"], row["prediction"]["scores"])]


_register_notebook_block("data")


In [ ]:
"""Kaggle에서 공유 Drive의 데이터·체크포인트를 이어 사용한다."""
import urllib.error
import urllib.parse
import urllib.request
from getpass import getpass

DRIVE_ACCESS_TOKEN = None
DRIVE_FOLDER_TYPE = "application/vnd.google-apps.folder"


def connect_drive():
    """토큰은 숨김 입력으로만 받아 현재 세션 메모리에 보관한다."""
    global DRIVE_ACCESS_TOKEN
    DRIVE_ACCESS_TOKEN = getpass("Google Drive Access token (공유 폴더 읽기·쓰기 권한): ").strip()
    if not DRIVE_ACCESS_TOKEN:
        raise ValueError("토큰이 비었습니다. 공유 폴더 권한이 있는 토큰을 숨김 입력하세요.")


def drive_request(url, data=None, method="GET", headers=None, raw=False):
    """Google Drive API에 인증하며 오류에 토큰을 포함하지 않는다."""
    if not DRIVE_ACCESS_TOKEN:
        raise RuntimeError("Drive 인증이 없습니다. connect_drive()를 먼저 실행하세요.")
    request = urllib.request.Request(url, data=data, method=method,
        headers={"Authorization": "Bearer " + DRIVE_ACCESS_TOKEN, **(headers or {})})
    try:
        response = urllib.request.urlopen(request, timeout=120)
    except urllib.error.HTTPError as error:
        if error.code == 308:
            return error
        raise RuntimeError(f"Drive HTTP {error.code}. 401이면 connect_drive()로 토큰을 갱신하고, "
                           "403이면 폴더·토큰의 읽기/쓰기 권한을 확인하세요.") from None
    if raw:
        return response
    with response:
        content = response.read()
        return json.loads(content) if content else {}


def drive_children(parent):
    """공유 폴더 자식 목록을 페이지 끝까지 읽는다."""
    rows, page = [], None
    while True:
        params = {"q": f"'{parent}' in parents and trashed = false", "pageSize": 1000,
                  "fields": "nextPageToken,files(id,name,mimeType,size,md5Checksum)",
                  "supportsAllDrives": "true", "includeItemsFromAllDrives": "true"}
        if page:
            params["pageToken"] = page
        result = drive_request("https://www.googleapis.com/drive/v3/files?" + urllib.parse.urlencode(params))
        rows.extend(result.get("files", []))
        page = result.get("nextPageToken")
        if not page:
            return rows


def drive_unique(parent, name):
    """동일 이름이 여러 개면 임의의 폴더나 체크포인트를 고르지 않는다."""
    matches = [row for row in drive_children(parent) if row["name"] == name]
    if len(matches) > 1:
        raise RuntimeError(f"Drive에 같은 이름이 여러 개 있습니다: {name}. 폴더를 확인하세요.")
    return matches[0] if matches else None


def drive_folder(parent, name):
    """필요한 결과 하위 폴더를 만들거나 기존 폴더를 사용한다."""
    found = drive_unique(parent, name)
    if found:
        if found["mimeType"] != DRIVE_FOLDER_TYPE:
            raise RuntimeError(f"폴더와 파일 이름이 충돌합니다: {name}. 결과 위치를 확인하세요.")
        return found["id"]
    result = drive_request("https://www.googleapis.com/drive/v3/files?supportsAllDrives=true",
        json.dumps({"name": name, "mimeType": DRIVE_FOLDER_TYPE, "parents": [parent]}).encode(),
        "POST", {"Content-Type": "application/json"})
    return result["id"]


def file_md5(path):
    """Drive가 제공하는 파일 전송 검증용 MD5를 계산한다."""
    digest = hashlib.md5()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def drive_download(item, destination):
    """완료 파일만 사용하고 서버 크기·체크섬을 검사한다."""
    destination = Path(destination)
    if item["name"] in ("", ".", "..") or Path(item["name"]).name != item["name"]:
        raise ValueError("Drive 파일명에 경로가 들어 있습니다. 공유 파일 이름을 확인하세요.")
    if destination.exists() and destination.stat().st_size == int(item.get("size", -1)):
        if item.get("md5Checksum") and file_md5(destination) == item["md5Checksum"]:
            return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".part")
    with drive_request(f"https://www.googleapis.com/drive/v3/files/{item['id']}?alt=media&supportsAllDrives=true",
                       raw=True) as response, temporary.open("wb") as stream:
        shutil.copyfileobj(response, stream, length=8 * 1024 * 1024)
    if temporary.stat().st_size != int(item["size"]) or (
            item.get("md5Checksum") and file_md5(temporary) != item["md5Checksum"]):
        raise IOError("Drive 다운로드가 불완전합니다. 같은 셀을 다시 실행하세요.")
    os.replace(temporary, destination)
    return destination


def drive_upload(path, parent):
    """큰 체크포인트도 분할 전송하고 원격 체크섬까지 확인한다."""
    path = Path(path)
    checksum, size = file_md5(path), path.stat().st_size
    existing = drive_unique(parent, path.name)
    if existing and existing.get("md5Checksum") == checksum:
        return existing
    metadata = {"name": path.name}
    endpoint = "https://www.googleapis.com/upload/drive/v3/files"
    if existing:
        endpoint += "/" + existing["id"]
    else:
        metadata["parents"] = [parent]
    endpoint += "?uploadType=resumable&supportsAllDrives=true"
    response = drive_request(endpoint, json.dumps(metadata).encode(),
        "PATCH" if existing else "POST", {"Content-Type": "application/json; charset=UTF-8",
        "X-Upload-Content-Type": "application/octet-stream", "X-Upload-Content-Length": str(size)}, raw=True)
    location = response.headers.get("Location")
    response.close()
    parsed = urllib.parse.urlparse(location or "")
    hostname = parsed.hostname or ""
    if parsed.scheme != "https" or not (hostname == "googleapis.com" or hostname.endswith(".googleapis.com")):
        raise RuntimeError("Drive 업로드 주소를 확인할 수 없습니다. 인증 후 다시 실행하세요.")
    result = None
    with path.open("rb") as stream:
        offset = 0
        while offset < size or (size == 0 and result is None):
            chunk = stream.read(8 * 1024 * 1024)
            content_range = f"bytes {offset}-{offset + len(chunk) - 1}/{size}" if size else "bytes */0"
            response = drive_request(location, chunk, "PUT", {"Content-Type": "application/octet-stream",
                "Content-Range": content_range, "Content-Length": str(len(chunk))}, raw=True)
            if response.code == 308:
                confirmed = response.headers.get("Range")
                if not confirmed:
                    raise IOError("Drive가 전송 위치를 돌려주지 않았습니다. 다시 동기화하세요.")
                next_offset = int(confirmed.split("-")[-1]) + 1
                if next_offset != offset + len(chunk):
                    raise IOError("Drive가 확인한 전송 위치가 다릅니다. 동기화 셀을 다시 실행하세요.")
                offset = next_offset
                response.close()
            else:
                with response:
                    result = json.load(response)
                break
    if not result:
        raise IOError("Drive 파일 업로드가 완료되지 않았습니다. 동기화를 다시 실행하세요.")
    uploaded = drive_request("https://www.googleapis.com/drive/v3/files/" + result["id"] +
        "?fields=id,name,size,md5Checksum&supportsAllDrives=true")
    if uploaded.get("md5Checksum") != checksum or int(uploaded.get("size", -1)) != size:
        raise IOError("Drive 업로드 검증이 실패했습니다. 현재 run을 종료하지 말고 동기화를 재시도하세요.")
    return uploaded


def download_team_zip(name):
    """Kaggle에서 지정된 공유 데이터 ZIP 하나를 내려받는다."""
    item = drive_unique(DATA_DRIVE_FOLDER_ID, name)
    if not item:
        raise FileNotFoundError(f"공유 데이터 폴더에 {name}이 없습니다. 데이터 링크를 확인하세요.")
    return drive_download(item, ZIP_ROOT / name)


def sync_run(run_dir, include_wandb=False):
    """Colab은 Drive 직접 저장, Kaggle은 변경 파일 업로드 완료까지 확인한다."""
    run_dir = Path(run_dir)
    if PLATFORM != "kaggle":
        # Colab 직접 저장 후 이전 Kaggle 전송 완료 표시는 만료된다.
        marker = run_dir / "sync_complete.json"
        if marker.exists():
            marker.unlink()
        return
    remote_root = drive_folder(RESULT_DRIVE_FOLDER_ID, run_dir.name)
    parents = {".": remote_root}
    files = [p for p in sorted(run_dir.rglob("*")) if p.is_file() and not p.is_symlink()
             and not p.name.startswith(".") and p.suffix != ".part"
             and (include_wandb or "wandb" not in p.relative_to(run_dir).parts)
             and p.name != "sync_complete.json"]
    verified = {}
    for path in files:
        relative = path.relative_to(run_dir)
        parent = Path(".")
        for part in relative.parent.parts:
            next_parent = parent / part
            if next_parent.as_posix() not in parents:
                parents[next_parent.as_posix()] = drive_folder(parents[parent.as_posix()], part)
            parent = next_parent
        uploaded = drive_upload(path, parents[parent.as_posix()])
        verified[relative.as_posix()] = {"size": int(uploaded["size"]), "md5": uploaded["md5Checksum"]}
    marker = {"time_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "files": verified}
    write_json(run_dir / "sync_complete.json", marker)
    drive_upload(run_dir / "sync_complete.json", remote_root)
    print("공유 Drive 동기화·체크섬 검증 완료:", run_dir.name, flush=True)


def restore_run_from_drive(run_name):
    """Kaggle 새 세션에서 공유 run을 임시 폴더에 내려받은 뒤 사용한다."""
    if PLATFORM != "kaggle" or Path(run_name).name != run_name or run_name not in {
            experiment_spec(v, e)["run_name"] for e in EPOCH_BUDGETS for v in VARIANTS} | {"portability_smoke_v2"}:
        raise ValueError("Kaggle의 9개 계획 중 run 이름을 지정하세요.")
    destination = RUN_ROOT / run_name
    if destination.exists():
        raise FileExistsError("로컬 run이 이미 있습니다. 실행 계획에서 상태를 확인하고 덮어쓰지 마세요.")
    remote = drive_unique(RESULT_DRIVE_FOLDER_ID, run_name)
    if not remote:
        raise FileNotFoundError("공유 Drive에 해당 run이 없습니다. run 이름을 확인하세요.")
    staging = RUN_ROOT / (".restore_" + run_name)
    staging.mkdir(parents=True, exist_ok=True)
    downloaded_names = set()
    def download_tree(folder, local):
        for item in drive_children(folder):
            if item["name"].startswith(".") or item["name"].endswith(".part"):
                continue
            if item["name"] in ("", ".", "..") or Path(item["name"]).name != item["name"]:
                raise ValueError("공유 폴더의 파일명에 경로가 있습니다. 파일 이름을 확인하세요.")
            target = local / item["name"]
            if not target.resolve().is_relative_to(staging.resolve()):
                raise ValueError("공유 파일이 복원 폴더 밖을 가리킵니다.")
            if item["mimeType"] == DRIVE_FOLDER_TYPE:
                download_tree(item["id"], target)
            else:
                drive_download(item, target)
                downloaded_names.add(target.relative_to(staging).as_posix())
    download_tree(remote["id"], staging)
    if not {"config.json", "weights/last.pt"}.issubset(downloaded_names):
        raise RuntimeError("공유 체크포인트가 불완전합니다. 원래 환경에서 저장·동기화 상태를 확인하세요.")
    marker_path = staging / "sync_complete.json"
    if marker_path.is_file():
        marker = json.loads(marker_path.read_text())
        for name, meta in marker["files"].items():
            path = (staging / name).resolve()
            if not path.is_relative_to(staging.resolve()):
                raise ValueError("동기화 기록에 잘못된 경로가 있습니다.")
            if not path.is_file() or file_md5(path) != meta["md5"]:
                raise RuntimeError("공유 run의 전송 완료 기록과 파일이 다릅니다. 원래 환경의 동기화를 완료하세요.")
    os.replace(staging, destination)
    print("공유 run 복원 완료:", destination)
    return destination


_register_notebook_block("storage")


In [ ]:
def seed_everything(seed, deterministic=True):
    """분할을 변경하지 않고 학습 난수만 설정한다.

    Args:
        seed (int): 학습 seed.
        deterministic (bool): 결정적 연산 사용 여부.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
        torch.cuda.manual_seed_all(seed)
    # 결정적 연산을 끄면 cudnn이 입력 크기별로 가장 빠른 알고리즘을 골라 쓴다(이 데이터의 모델 입력은 512×640 하나).
    torch.backends.cudnn.benchmark = not deterministic
    torch.backends.cudnn.deterministic = deterministic
    torch.use_deterministic_algorithms(deterministic)


def _frozen_normalization(module):
    for name, child in list(module.named_children()):
        if isinstance(child, torch.nn.BatchNorm2d):
            frozen = FrozenBatchNorm2d(child.num_features, eps=0.0)
            for attribute in ("weight", "bias", "running_mean", "running_var"):
                getattr(frozen, attribute).copy_(getattr(child, attribute).detach())
            setattr(module, name, frozen)
        elif isinstance(child, FrozenBatchNorm2d):
            child.eps = 0.0
        else:
            _frozen_normalization(child)


def build_model(num_classes, spec, config, load_coco=True):
    """같은 COCO 검출 모델에서 출력층과 학습 범위를 설정한다.

    Args:
        num_classes (int): 배경을 제외한 클래스 수.
        spec (dict): 실험 조건.
        config (dict): 공통 설정.
        load_coco (bool): 신규 학습은 True. 저장된 전체 체크포인트 복원 시에만 False.

    Returns:
        Module: CPU의 Faster R-CNN. 모든 정규화 통계·계수는 고정.
    """
    validate_config(config)
    validate_spec(spec)
    weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1 if load_coco else None
    model = fasterrcnn_resnet50_fpn(weights=weights, weights_backbone=None,
                                    trainable_backbone_layers=5 if load_coco else None,
                                    min_size=config["image_size"], max_size=config["image_size"])
    _frozen_normalization(model)
    features = model.roi_heads.box_predictor.cls_score.in_features
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(spec["seed"])
        model.roi_heads.box_predictor = FastRCNNPredictor(features, num_classes + 1)
    for parameter in model.parameters():
        parameter.requires_grad_(True)
    model.roi_heads.score_thresh = config["score_threshold"]
    model.roi_heads.nms_thresh = config["nms_iou"]
    model.roi_heads.detections_per_img = config["max_detections"]
    return model


def parameter_summary(model):
    """실제 학습 모듈과 파라미터 수를 확인한다.

    Args:
        model (Module): 검사 모델.

    Returns:
        DataFrame: 모듈별 전체·학습 가능한 파라미터 수.
    """
    groups = {}
    for name, parameter in model.named_parameters():
        group = ".".join(name.split(".")[:3]) if name.startswith("backbone.body") else name.split(".")[0]
        row = groups.setdefault(group, {"module": group, "total": 0, "trainable": 0})
        row["total"] += parameter.numel()
        row["trainable"] += parameter.numel() if parameter.requires_grad else 0
    return pd.DataFrame(groups.values())


def make_optimizer(model, spec, config):
    """고정 파라미터를 제외하고 AdamW decay 그룹을 만든다.

    Args:
        model (Module): 학습 모델.
        spec (dict): 학습률 조건.
        config (dict): AdamW 설정.

    Returns:
        AdamW: bias·1차원 파라미터에는 decay 미적용.
    """
    groups = [[], []]
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            groups[int(name.endswith("bias") or parameter.ndim <= 1)].append(parameter)
    return torch.optim.AdamW([{"params": groups[0], "weight_decay": config["weight_decay"]},
                               {"params": groups[1], "weight_decay": 0.0}],
                              lr=spec["lr"], betas=tuple(config["betas"]))


def make_scheduler(optimizer, steps_per_epoch, config):
    """optimizer step 단위의 선형 warmup·cosine 스케줄러를 만든다.

    Args:
        optimizer (Optimizer): AdamW.
        steps_per_epoch (int): 누적 후 epoch당 예정 optimizer 갱신 수.
        config (dict): 학습 예산·warmup 설정.

    Returns:
        LambdaLR: 첫 step부터 warmup을 적용하는 scheduler.
    """
    total = config["max_epochs"] * steps_per_epoch
    warmup = int(config["warmup_epochs"] * steps_per_epoch)

    def factor(step):
        if warmup and step < warmup:
            return (step + 1) / warmup
        progress = min(1.0, max(0.0, (step-warmup) / max(1, total-warmup-1)))
        return config["final_lr_ratio"] + (1-config["final_lr_ratio"]) * (1+math.cos(math.pi*progress))/2

    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def _loader(dataset, config, training=False, generator=None):
    return DataLoader(dataset, batch_size=config["batch_size"] if training else config["inference_batch_size"],
                      shuffle=training, num_workers=LOADER_WORKERS,
                      collate_fn=collate_detection, generator=generator,
                      pin_memory=torch.cuda.is_available())


def _synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize()


def collect_predictions(model, dataset, config, reverse, with_targets=True):
    """추론을 한 번 수행해 원본 category ID의 공통 결과를 만든다.

    Args:
        model (Module): 복원 또는 학습 모델.
        dataset (PillDataset): 순서가 고정된 이미지 목록.
        config (dict): 추론 설정.
        reverse (dict): 모델 번호→원래 category ID.
        with_targets (bool): validation 정답 포함 여부.

    Returns:
        tuple: 공통 결과, 순수 추론 시간·전체 시간.
    """
    device = next(model.parameters()).device
    records, model_seconds = [], 0.0
    start_all = time.perf_counter()
    model.eval()
    with torch.inference_mode():
        for images, targets in _loader(dataset, config):
            inputs = to_model_inputs(images, device)
            _synchronize(device)
            start = time.perf_counter()
            outputs = model(inputs)
            _synchronize(device)
            model_seconds += time.perf_counter()-start
            for output, target in zip(outputs, targets):
                info = dataset.frame.iloc[len(records)].to_dict()
                prediction = {key: output[key].detach().cpu() for key in ("boxes", "labels", "scores")}
                prediction["labels"] = torch.tensor([reverse[int(c)] for c in prediction["labels"]], dtype=torch.int64)
                record = {key: int(info[key]) for key in ("image_id", "width", "height")}
                record.update(file_name=info["file_name"], image_path=str(dataset.image_dir / info["file_name"]),
                              prediction=prediction)
                if with_targets:
                    record["target"] = {"boxes": target["boxes"].cpu(),
                                        "labels": torch.tensor([reverse[int(c)] for c in target["labels"]], dtype=torch.int64)}
                records.append(record)
    return records, {"model_ms_per_image": 1000*model_seconds/max(1, len(records)),
                     "wall_seconds": time.perf_counter()-start_all, "images": len(records)}


def _make_scaler(config, device):
    """CUDA AMP 설정에 맞는 손실 스케일러를 만든다.

    Args:
        config (dict): 정밀도 설정.
        device (torch.device): 학습 장치.

    Returns:
        GradScaler: FP32에서는 비활성화한 스케일러.
    """
    enabled = config["precision"] == "amp_fp16"
    if enabled and device.type != "cuda":
        raise RuntimeError("AMP FP16 학습에는 CUDA가 필요합니다. 현재 환경의 CUDA GPU를 선택하세요.")
    return torch.amp.GradScaler("cuda", enabled=enabled)


def _batch_policy(config):
    """실제 배치와 누적 후 목표 배치를 구분해 기록한다.

    Args:
        config (dict): 실제 배치·누적 횟수 설정.

    Returns:
        dict: 단일 GPU 기준 실제 배치·누적 횟수·목표 실효 배치.
    """
    return {"micro_batch_size": config["batch_size"],
            "accumulation_steps": config["accumulation_steps"],
            "effective_batch_size": config["batch_size"] * config["accumulation_steps"]}


def _accumulation_groups(loader, steps):
    """CPU의 작은 배치를 묶으며 마지막 불완전한 묶음도 반환한다.

    Args:
        loader (DataLoader): 원본 CPU 이미지와 정답을 읽는 로더.
        steps (int): 가중치 갱신 전 누적할 최대 작은 배치 수.

    Yields:
        list: GPU로 한 번에 올리지 않을 작은 배치 목록.
    """
    group = []
    for batch in loader:
        group.append(batch)
        if len(group) == steps:
            yield group
            group = []
    if group:
        yield group


def _training_step(model, optimizer, scaler, batches, config):
    """작은 배치별 역전파 후 누적 묶음마다 한 번만 가중치를 갱신한다.

    Args:
        model (Module): 학습 모드의 검출 모델.
        optimizer (Optimizer): 학습 optimizer.
        scaler (GradScaler): 묶음 전체에서 같은 scale을 유지할 스케일러.
        batches (list): CPU 이미지·정답의 작은 배치 목록.
        config (dict): 학습 정밀도·누적 횟수 설정.

    Returns:
        tuple: 실제 이미지 수로 가중 평균한 세부 loss와 갱신 성공 여부.
    """
    if not batches or len(batches) > config["accumulation_steps"]:
        raise ValueError("누적 묶음 크기가 잘못됐습니다. accumulation_steps와 로더를 확인하세요.")
    total_images = sum(len(images) for images, _ in batches)
    if total_images == 0:
        raise ValueError("빈 학습 묶음입니다. 고정 학습 분할을 확인하세요.")
    device = next(model.parameters()).device
    optimizer.zero_grad(set_to_none=True)
    means = {}
    for images, targets in batches:
        if not images or len(images) != len(targets) or len(images) > config["batch_size"]:
            raise ValueError("작은 배치의 이미지·정답 수가 잘못됐습니다. batch_size와 로더를 확인하세요.")
        inputs = to_model_inputs(images, device)
        truth = [{key: value.to(device) for key, value in target.items()} for target in targets]
        weight = len(images) / total_images
        with torch.autocast(device_type=device.type, dtype=torch.float16,
                            enabled=config["precision"] == "amp_fp16"):
            losses = model(inputs, truth)
            # 마지막 묶음이 16개 미만이어도 실제 이미지 수로 평균한다.
            # 검출 loss의 내부 정규화·RoI 샘플링 때문에 물리 배치 16과 완전히 같지는 않다.
            loss = sum(losses.values()) * weight
        if not torch.isfinite(loss):
            raise FloatingPointError("학습 loss가 유한하지 않습니다. 이미지·라벨·학습률을 확인하세요.")
        scaler.scale(loss).backward()
        for key, value in losses.items():
            means[key] = means.get(key, 0.0) + float(value.detach()) * weight
        # 이전 작은 배치의 GPU 입력과 그래프를 다음 forward까지 붙잡지 않는다.
        del inputs, truth, losses, loss
    old_scale = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    updated = not scaler.is_enabled() or scaler.get_scale() >= old_scale
    return means, updated


@contextlib.contextmanager
def _validation_loss_context(model, seed):
    """검출 loss 경로만 켜고 모드·난수·정규화 버퍼를 보존한다.

    Args:
        model (Module): 현재 epoch의 모델.
        seed (int): RPN·RoI 표본 추출에 사용할 고정 seed.
    """
    modes = {module: module.training for module in model.modules()}
    buffers = {name: value.detach().clone() for name, value in model.named_buffers()}
    python_state, numpy_state = random.getstate(), np.random.get_state()
    device = next(model.parameters()).device
    devices = [device.index if device.index is not None else torch.cuda.current_device()] if device.type == "cuda" else []
    try:
        with torch.random.fork_rng(devices=devices):
            random.seed(seed)
            np.random.seed(seed)
            torch.random.default_generator.manual_seed(seed)
            if device.type == "cuda":
                with torch.cuda.device(device):
                    torch.cuda.manual_seed(seed)
            model.train()
            for module in model.modules():
                if isinstance(module, (torch.nn.modules.batchnorm._BatchNorm,
                                       torch.nn.modules.dropout._DropoutNd)):
                    module.eval()
            with torch.no_grad(), torch.autocast(device_type=device.type, enabled=False):
                yield
    finally:
        changed = False
        with torch.no_grad():
            for name, value in model.named_buffers():
                if not torch.equal(value, buffers[name]):
                    changed = True
                    value.copy_(buffers[name])
        for module, training in modes.items():
            module.training = training
        random.setstate(python_state)
        np.random.set_state(numpy_state)
        if changed:
            raise RuntimeError("validation loss 측정 중 버퍼가 바뀌어 복구했습니다. 모델 정규화 설정을 확인하세요.")


def validation_loss(model, dataset, config):
    """가중치 갱신 없이 validation 전체의 FP32 loss를 측정한다.

    Args:
        model (Module): 현재 epoch의 학습 모델.
        dataset (Dataset): 고정 validation 데이터.
        config (dict): 측정 batch와 고정 난수 설정.

    Returns:
        dict: 이미지 평균 전체 loss와 네 세부 loss.
    """
    device = next(model.parameters()).device
    loader = DataLoader(dataset, batch_size=config["val_loss_batch_size"], shuffle=False,
                        num_workers=LOADER_WORKERS, collate_fn=collate_detection,
                        generator=torch.Generator().manual_seed(config["loss_measurement_seed"]),
                        pin_memory=device.type == "cuda")
    totals, seen = {}, 0
    expected = {"loss_classifier", "loss_box_reg", "loss_objectness", "loss_rpn_box_reg"}
    with _validation_loss_context(model, config["loss_measurement_seed"]):
        for images, targets in loader:
            values = model(to_model_inputs(images, device),
                           [{key: value.to(device) for key, value in target.items()} for target in targets])
            if set(values) != expected:
                raise RuntimeError("검출 loss 네 종류가 필요합니다. Faster R-CNN 모델 구성을 확인하세요.")
            for key, value in values.items():
                if value.numel() != 1 or not torch.isfinite(value):
                    raise FloatingPointError("validation loss가 유한하지 않습니다. 라벨·모델을 확인하세요.")
                totals[key] = totals.get(key, 0.0) + float(value.detach()) * len(images)
            seen += len(images)
    if seen != len(dataset) or seen == 0:
        raise RuntimeError("validation 전체를 측정하지 못했습니다. 고정 분할을 확인하세요.")
    components = {f"val_{key}": value / seen for key, value in totals.items()}
    return {"val_loss": sum(components.values()), **components}


"""2차 9개 독립 실험, 매 epoch 저장, 동일 조건 재개."""
EFFECTIVE_BATCH = 16  # 실제 batch × 누적. 실제 batch는 1번 셀 MICRO_BATCH
DEFAULT_CONFIG = {
    "max_epochs": 5, "patience": 30, "image_size": 640,
    "batch_size": 16, "accumulation_steps": 1, "weight_decay": 1e-4,
    "betas": [0.9, 0.999], "warmup_epochs": 3, "final_lr_ratio": 0.01,
    "score_threshold": 0.001, "nms_iou": 0.5,
    "max_detections": 100, "preview_score": 0.30, "preview_count": 3,
    "wandb_project": "pill-object-detection", "wandb_entity": "codeit-prj1-team2",
    "wandb_mode": "online", "use_wandb": True, "require_cuda": True,
    "deterministic": False, "precision": "amp_fp16",
    "normalization": "FrozenBatchNorm2d_eps0_fixed_statistics_and_affine",
    "augmentation": "none", "weights": "COCO_V1", "optimizer": "AdamW",
    "learning_rate": 3e-4, "learning_rates": [3e-4],
    "val_loss_batch_size": 8, "inference_batch_size": 8, "loss_measurement_seed": 0,
    "inference_precision": "fp32", "val_loss_precision": "fp32",
}
# 이미지 디코드를 GPU 계산과 겹친다. 읽는 이미지와 순서는 worker 수와 상관없어서 실험 식별자 대신 환경 이력에 남긴다.
LOADER_WORKERS = 0 if PLATFORM == "local" else min(8, max(1, (os.cpu_count() or 2) - 1))
SMOKE_REPORTS = {}


def validate_config(config):
    """승인한 고정 학습 조건을 검사한다."""
    if set(config) != set(DEFAULT_CONFIG):
        raise ValueError("설정 키가 기본값과 다릅니다. DEFAULT_CONFIG에서 복사해 지정하세요.")
    for key, expected in DEFAULT_CONFIG.items():
        if key not in ("max_epochs", "batch_size", "accumulation_steps") and config[key] != expected:
            raise ValueError(f"확정한 학습 조건이 다릅니다: {key}={config[key]!r}. 기본 설정으로 되돌리세요.")
    batch, steps = config["batch_size"], config["accumulation_steps"]
    if type(batch) is not int or type(steps) is not int or batch not in (16, 8, 4, 2) or batch * steps != EFFECTIVE_BATCH:
        raise ValueError("실제 batch는 16·8·4·2 중 하나, 실효 batch는 16이어야 합니다. 1번 셀의 MICRO_BATCH를 확인하세요.")
    if type(config["max_epochs"]) is not int or config["max_epochs"] not in (5, 10, 20):
        raise ValueError("총 epoch는 5·10·20 중 하나여야 합니다. 실험 계획을 확인하세요.")


def experiment_spec(variant, epochs):
    """방법·독립 학습 예산·고정 seed 0으로 실행을 구분한다."""
    if variant not in ("v2", "v3", "v4") or type(epochs) is not int or epochs not in (5, 10, 20):
        raise ValueError("방법은 v2~v4, epoch는 5·10·20으로 지정하세요.")
    return {"variant": variant, "epochs": epochs, "seed": 0, "mode": "ft", "lr": 3e-4,
            "run_name": f"{variant}_frcnn_e{epochs}_s0"}


def validate_spec(spec):
    """실행 이름과 실제 학습 조건의 불일치를 막는다."""
    if spec != experiment_spec(spec["variant"], spec["epochs"]):
        raise ValueError("실행 정보가 계획과 다릅니다. experiment_spec으로 다시 만드세요.")


def config_for(spec):
    """독립 실험은 총 epoch와 GPU에 맞춘 실제 batch만 정하고 다른 학습 조건은 유지한다."""
    validate_spec(spec)
    config = copy.deepcopy(DEFAULT_CONFIG)
    config["max_epochs"] = spec["epochs"]
    # 실효 batch 16은 그대로 두고 실제 batch와 누적 횟수만 나눈다.
    config["batch_size"] = MICRO_BATCH
    config["accumulation_steps"] = EFFECTIVE_BATCH // MICRO_BATCH
    validate_config(config)
    return config


def experiment_identity(bundle, config, spec):
    """경로·GPU 이름을 제외하고 데이터·학습 조건·실행 코드·호환 버전을 검증한다."""
    validate_config(config); validate_spec(spec)
    if bundle["variant"] != spec["variant"] or config["max_epochs"] != spec["epochs"]:
        raise ValueError("데이터 방법과 학습 예산이 다릅니다. 선택한 run을 확인하세요.")
    recipe = {"spec": spec, "config": config, "data_identity": bundle["identity"],
              "class_mapping": {str(k): v for k, v in bundle["category_to_label"].items()},
              "code": code_identity(), "python": list(sys.version_info[:2]), "packages": package_versions()}
    return stable_hash(recipe), recipe


def _device(config):
    """승인된 학습은 CUDA GPU에서만 시작한다."""
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU가 없습니다. Colab/Kaggle GPU와 커널을 확인하세요.")
    return torch.device("cuda:0")


def _atomic_save(path, value):
    """완성된 체크포인트만 교체한다."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".part")
    torch.save(value, temporary)
    os.replace(temporary, path)


def _rng_state(generator):
    """실제로 사용하는 단일 GPU의 난수 상태만 보관한다."""
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(), "loader": generator.get_state(),
            "cuda": torch.cuda.get_rng_state(0) if torch.cuda.is_available() else None}


def _restore_rng(state, generator):
    """장착된 전체 GPU 개수가 달라도 사용 GPU 0의 상태를 복원한다."""
    random.setstate(state["python"]); np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"]); generator.set_state(state["loader"])
    if state["cuda"] is not None:
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA 학습 체크포인트입니다. GPU 환경에서 재개하세요.")
        torch.cuda.set_rng_state(state["cuda"], 0)


def _smoke_key(bundle, config):
    """총 epoch와 무관한 1회 갱신 검사는 같은 방법의 예산 사이에서 재사용한다."""
    return stable_hash({"data": bundle["identity"], "config": {k: v for k, v in config.items()
                         if k != "max_epochs"}, "code": code_identity(), "packages": package_versions()})


def smoke_test(bundle, config, spec):
    """실제 batch·AMP·누적·검증 loss·저장 복원을 짧게 확인하고 GPU 메모리 최대치를 기록한다."""
    validate_config(config)
    device = _device(config)
    torch.cuda.reset_peak_memory_stats()
    seed_everything(0, config["deterministic"])
    generator = torch.Generator().manual_seed(0)
    dataset = make_dataset(bundle, "train")
    count = config["batch_size"] * config["accumulation_steps"]
    if len(dataset) < count:
        raise ValueError("스모크용 이미지 16장이 부족합니다. 데이터 구성을 확인하세요.")
    sample = Subset(dataset, torch.randperm(len(dataset), generator=generator)[:count].tolist())
    # 본 학습과 같은 실제 batch를 쓴다.
    loader = DataLoader(sample, batch_size=config["batch_size"], shuffle=False, num_workers=0,
                        collate_fn=collate_detection)
    model = build_model(len(bundle["class_map"]), spec, config).to(device)
    optimizer = make_optimizer(model, spec, config)
    scaler = _make_scaler(config, device)
    try:
        batches = next(_accumulation_groups(loader, config["accumulation_steps"]))
        before = model.roi_heads.box_predictor.cls_score.weight.detach().clone()
        updated = False
        for attempt in range(8):
            model.train()
            losses, updated = _training_step(model, optimizer, scaler, batches, config)
            if updated:
                break
        if not updated or torch.equal(before, model.roi_heads.box_predictor.cls_score.weight.detach()):
            raise RuntimeError("AMP 누적 갱신 검사를 통과하지 못했습니다. GPU·loss·scale을 확인하세요.")
        optimizer.zero_grad(set_to_none=True)
        measured = validation_loss(model, Subset(make_dataset(bundle, "val"), [0, 1]), config)
        with tempfile.TemporaryDirectory() as temporary:
            checkpoint = Path(temporary) / "smoke.pt"
            _atomic_save(checkpoint, {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                                     "scaler": scaler.state_dict(), "epoch": 0})
            restored = torch.load(checkpoint, map_location="cpu", weights_only=False)
            if not torch.isfinite(restored["model"]["roi_heads.box_predictor.cls_score.weight"]).all():
                raise RuntimeError("스모크 체크포인트가 유효하지 않습니다. 저장 환경을 확인하세요.")
        report = {"passed": True, "key": _smoke_key(bundle, config), "variant": bundle["variant"],
                  "images": count, "micro_batches": len(batches), "successful_updates": 1,
                  "amp_retries": attempt, "losses": losses, "validation": measured,
                  "peak_gpu_gb": torch.cuda.max_memory_allocated() / 1024 ** 3, "environment": runtime_info()}
        SMOKE_REPORTS[report["key"]] = report
        print(f"짧은 검사 통과: 실제 batch {config['batch_size']} × 누적 {config['accumulation_steps']}, "
              f"GPU 메모리 최대 {report['peak_gpu_gb']:.1f}GB. 검사 가중치는 본 학습에 사용하지 않습니다.")
        return report
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError(f"실제 batch {config['batch_size']}에서 GPU 메모리가 부족합니다. 1번 셀의 MICRO_BATCH를 "
                           "절반(16→8)으로 바꿔 실행한 뒤 학습 셀을 다시 실행하세요.") from error
    finally:
        del model, optimizer, scaler
        gc.collect(); torch.cuda.empty_cache()


def read_checkpoint(run_dir, signature):
    """현재 파일이 불완전하면 이전 정상 epoch와 짝이 맞는 best를 찾는다."""
    run_dir = Path(run_dir)
    failures = []
    for name in ("last.pt", "previous_last.pt"):
        path = run_dir / "weights" / name
        if not path.is_file():
            continue
        try:
            state = torch.load(path, map_location="cpu", weights_only=False)
            if state.get("signature") != signature:
                raise ValueError("실험 식별자가 다릅니다")
            required = {"model", "optimizer", "scheduler", "scaler", "rng", "epoch", "history", "best_epoch"}
            if not required.issubset(state) or not state["history"] or state["history"][-1]["epoch"] != state["epoch"]:
                raise ValueError("재개 상태·학습 이력이 불완전합니다")
            if not all(torch.isfinite(t).all() for t in state["model"].values() if t.is_floating_point()):
                raise ValueError("모델에 유한하지 않은 값이 있습니다")
            for best_name in ("best.pt", "previous_best.pt"):
                best_path = run_dir / "weights" / best_name
                if not best_path.is_file():
                    continue
                try:
                    best = torch.load(best_path, map_location="cpu", weights_only=False)
                except Exception:
                    continue
                if best.get("signature") == signature and best["epoch"] == state["best_epoch"]:
                    return state, path, best_path
            raise ValueError("현재 epoch와 일치하는 best 체크포인트가 없습니다")
        except Exception as error:
            failures.append(f"{name}: {type(error).__name__}")
    raise RuntimeError("사용할 정상 체크포인트가 없습니다. 공유 저장을 확인하세요. " + "; ".join(failures))


def experiment_plan():
    """epoch 우선 순서와 완료·새 학습·명시적 재개 상태를 표시한다."""
    rows = []
    for epochs in EPOCH_BUDGETS:
        for variant in VARIANTS:
            spec = experiment_spec(variant, epochs)
            config = config_for(spec)
            root = RUN_ROOT / spec["run_name"]
            status = "새 COCO 학습"
            if root.exists():
                status = "분석·보존 완료" if (root / "analysis_complete.json").is_file() else (
                    "학습 완료·분석 필요" if (root / "result.json").is_file() else "재개 승인 필요")
            rows.append({**spec, "status": status, "resume_selected": spec["run_name"] in RESUME_RUN_NAMES,
                         "batch": config["batch_size"], "accumulation": config["accumulation_steps"],
                         "run_dir": str(root)})
    return pd.DataFrame(rows)


def run_experiment(bundle, spec, resume=False):
    """독립 학습·동일 조건 재개를 수행하고 매 epoch 공식 val로 best를 선택한다."""
    config = config_for(spec)
    signature, recipe = experiment_identity(bundle, config, spec)
    run_dir = RUN_ROOT / spec["run_name"]
    config_path = run_dir / "config.json"
    if run_dir.exists():
        if not config_path.is_file():
            raise RuntimeError("기존 결과 폴더에 config.json이 없습니다. 덮어쓰지 말고 경로를 확인하세요.")
        manifest = json.loads(config_path.read_text())
        if manifest["signature"] != signature:
            raise ValueError("기존 run의 데이터·설정·코드·패키지가 다릅니다. 같은 실행으로 재개할 수 없습니다. "
                             "MICRO_BATCH가 기존 run과 같은지 확인하고, 이전 코드로 만든 폴더면 이름을 바꿔 보관한 뒤 새로 시작하세요.")
        if (run_dir / "result.json").is_file():
            print("완료된 학습 재사용:", spec["run_name"])
            return run_dir
        if not resume:
            raise RuntimeError(f"미완료 run이 있습니다: {spec['run_name']}. RESUME_RUN_NAMES에 명시하세요.")
    elif resume:
        raise FileNotFoundError("재개할 로컬 run이 없습니다. Kaggle이면 공유 run 복원 셀을 먼저 실행하세요.")
    device = _device(config)
    key = _smoke_key(bundle, config)
    smoke = SMOKE_REPORTS.get(key) or smoke_test(bundle, config, spec)
    state = source_state = source_best = None
    if resume:
        if any((run_dir / "weights" / name).is_file() for name in ("last.pt", "previous_last.pt")):
            state, source_state, source_best = read_checkpoint(run_dir, signature)
        elif (run_dir / "metrics.csv").exists() or any((run_dir / "weights").glob("*.pt")):
            raise RuntimeError("완료 기록은 있는데 last 체크포인트가 없습니다. 공유 저장을 확인하세요.")
        else:
            print("첫 epoch 저장 전 중단: 같은 초기화와 run ID로 다시 시작합니다.")
    else:
        run_dir.mkdir(parents=True, exist_ok=False)
        (run_dir / "weights").mkdir()
        manifest = {"signature": signature, "recipe": recipe, "spec": spec, "config": config,
                    "run_id": uuid.uuid4().hex[:12], "run_name": spec["run_name"],
                    "source": "new_COCO_initialization"}
        write_json(config_path, manifest)
        write_json(run_dir / "data_fingerprint.json", bundle["fingerprint"])
        write_frame(run_dir / "class_map.csv", bundle["class_map"])
        for name in ("dataset_info.json", "classes.csv"):
            shutil.copy2(bundle["root"] / name, run_dir / name)
        shutil.copy2(bundle["root"] / "eval/val_gt.json", run_dir / "val_gt.json")
    write_json(run_dir / "smoke.json", smoke)
    history_path = run_dir / "environment_history.json"
    environments = json.loads(history_path.read_text()) if history_path.exists() else []
    environments.append({**runtime_info(), "resume": resume, "loader_workers": LOADER_WORKERS,
                         "checkpoint": source_state.name if source_state else None})
    write_json(history_path, environments)
    seed_everything(0, config["deterministic"])
    generator = torch.Generator().manual_seed(0)
    loader = _loader(make_dataset(bundle, "train"), config, True, generator)
    model = build_model(len(bundle["class_map"]), spec, config, load_coco=state is None).to(device)
    optimizer = make_optimizer(model, spec, config)
    steps_per_epoch = math.ceil(len(loader) / config["accumulation_steps"])
    scheduler = make_scheduler(optimizer, steps_per_epoch, config)
    scaler = _make_scaler(config, device)
    history, best, best_epoch, elapsed_before = [], -1.0, 0, 0.0
    if state:
        model.load_state_dict(state["model"])
        scheduler.load_state_dict(state["scheduler"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        history, best, best_epoch = state["history"], state["best"], state["best_epoch"]
        elapsed_before = state["training_seconds"]
        if source_state.name != "last.pt":
            _atomic_save(run_dir / "weights/last.pt", state)
        if source_best.name != "best.pt":
            best_saved = torch.load(source_best, map_location="cpu", weights_only=False)
            _atomic_save(run_dir / "weights/best.pt", best_saved)
        print(f"{spec['run_name']}: {state['epoch']} epoch 완료 상태에서 재개", flush=True)
    import wandb
    run = None
    started = time.perf_counter()
    try:
        run = wandb.init(entity=config["wandb_entity"], project=config["wandb_project"],
            id=manifest["run_id"], name=spec["run_name"], resume=("must" if state else "allow") if resume else "never",
            group=f"{spec['variant']}_frcnn", tags=["exp2", spec["variant"], f"e{spec['epochs']}", "seed0"],
            config={"training": config, "experiment": spec, "data_identity": bundle["identity"],
                    "dataset_info": bundle["dataset_info"], "class_mapping": recipe["class_mapping"],
                    "code": recipe["code"], "environment": environments[-1]}, dir=str(run_dir))
        if run is None or run.entity != config["wandb_entity"] or run.project != config["wandb_project"]:
            raise RuntimeError("W&B 팀 프로젝트 기록이 시작되지 않았습니다. entity·계정 권한을 확인하세요.")
        run.define_metric("epoch")
        run.define_metric("*", step_metric="epoch")
        if state:
            _restore_rng(state["rng"], generator)
        else:
            seed_everything(0, config["deterministic"])
        torch.cuda.reset_peak_memory_stats()
        for epoch in range((state["epoch"] + 1) if state else 1, config["max_epochs"] + 1):
            tick = time.perf_counter()
            model.train()
            totals, seen, updates, skipped, micro_batches = {}, 0, 0, 0, 0
            first_lr = optimizer.param_groups[0]["lr"]
            for batches in _accumulation_groups(loader, config["accumulation_steps"]):
                group_images = sum(len(images) for images, _ in batches)
                losses, updated = _training_step(model, optimizer, scaler, batches, config)
                if updated:
                    updates += 1; scheduler.step()
                else:
                    skipped += 1
                for key, value in losses.items():
                    totals[key] = totals.get(key, 0.0) + value * group_images
                seen += group_images; micro_batches += len(batches)
                if (updates + skipped) % 100 == 0:
                    print(f"  {epoch} epoch | {updates + skipped}/{steps_per_epoch} step | "
                          f"{seen / (time.perf_counter() - tick):.1f} 장/초", flush=True)
            train_seconds = time.perf_counter() - tick
            optimizer.zero_grad(set_to_none=True)
            if seen != len(loader.dataset) or updates == 0:
                raise RuntimeError("학습 전체를 처리하거나 가중치를 갱신하지 못했습니다. 데이터·AMP 상태를 확인하세요.")
            measured = validation_loss(model, make_dataset(bundle, "val"), config)
            records, speed = collect_predictions(model, make_dataset(bundle, "val"), config,
                                                  bundle["label_to_category"], with_targets=False)
            scores = evaluate_val(predictions_for_team(records), bundle["gt"])
            if any(scores[k] is None or not math.isfinite(scores[k]) for k in ("val", "kaggle", "test_only")):
                raise RuntimeError("공식 세 점수가 유효하지 않습니다. 전체 val·정답 연결을 확인하세요.")
            improved = scores["val"] > best
            if improved:
                best, best_epoch = scores["val"], epoch
            row = {"epoch": epoch, "train_loss": sum(totals.values()) / seen, **measured,
                   **{f"train_{k}": v / seen for k, v in totals.items()},
                   **{k: scores[k] for k in ("val", "kaggle", "test_only")},
                   "lr": first_lr, "last_lr": optimizer.param_groups[0]["lr"],
                   "train_seconds": train_seconds, "epoch_seconds": time.perf_counter() - tick,
                   "inference_ms": speed["model_ms_per_image"],
                   "peak_gpu_gb": torch.cuda.max_memory_allocated() / 1024 ** 3,
                   "optimizer_steps": updates, "amp_skipped_steps": skipped,
                   "micro_batches": micro_batches, "train_images": seen, "last_group_images": group_images,
                   "amp_scale": scaler.get_scale(), **_batch_policy(config)}
            history.append(row)
            weights = run_dir / "weights"
            if improved:
                if (weights / "best.pt").is_file():
                    shutil.copy2(weights / "best.pt", weights / "previous_best.pt")
                _atomic_save(weights / "best.pt", {"signature": signature, "epoch": epoch,
                    "model": model.state_dict(), "scores": scores, "config": config, "spec": spec})
            checkpoint = {"signature": signature, "epoch": epoch, "model": model.state_dict(),
                "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "rng": _rng_state(generator), "best": best,
                "best_epoch": best_epoch, "history": history, "config": config, "spec": spec,
                "run_id": manifest["run_id"], "training_seconds": elapsed_before + time.perf_counter() - started}
            if (weights / "last.pt").is_file():
                shutil.copy2(weights / "last.pt", weights / "previous_last.pt")
            _atomic_save(weights / "last.pt", checkpoint)
            write_frame(run_dir / "metrics.csv", pd.DataFrame(history))
            sync_run(run_dir)
            # 숫자 기록 실패는 숨기지 않는다. 체크포인트는 먼저 보존했다.
            run.log({**row, **{f"epoch_scores/{k}": scores[k] for k in ("val", "kaggle", "test_only")}})
            if config["preview_count"]:
                try:
                    diagnostic = diagnostic_records(predictions_for_team(records), bundle, "val")
                    images = []
                    for record in diagnostic[:config["preview_count"]]:
                        truth = record["target"]
                        picture = draw_boxes(record["image_path"],
                            [xyxy_to_xywh(b.tolist()) for b in truth["boxes"]],
                            [str(int(c)) for c in truth["labels"]], color="lime")
                        pred = record["prediction"]; keep = torch.where(pred["scores"] >= 0.30)[0][:20]
                        picture = draw_boxes(picture, [xyxy_to_xywh(pred["boxes"][i].tolist()) for i in keep],
                            [f"{int(pred['labels'][i])}:{float(pred['scores'][i]):.2f}" for i in keep], color="red")
                        picture.thumbnail((1000, 1000)); images.append(wandb.Image(picture, caption=f"epoch {epoch}"))
                    run.log({"epoch": epoch, "Media/validation": images})
                except Exception as error:
                    print("미리보기 이미지 기록 실패:", type(error).__name__, "숫자·체크포인트는 보존됨")
            _restore_rng(checkpoint["rng"], generator)
            print(f"{spec['run_name']} | {epoch}/{spec['epochs']} | train {row['train_loss']:.4f} "
                  f"val_loss {row['val_loss']:.4f} | val {scores['val']:.4f} "
                  f"kaggle_val {scores['kaggle']:.4f} test_only_val {scores['test_only']:.4f}", flush=True)
            del records, checkpoint
        if not history:
            raise RuntimeError("완료된 epoch가 없습니다. 체크포인트를 확인하세요.")
        result = {**spec, "signature": signature, "best_epoch": best_epoch, "last_epoch": history[-1]["epoch"],
                  "best_val": best, "training_seconds": elapsed_before + time.perf_counter() - started,
                  "state": "training_complete", "run_id": manifest["run_id"]}
        write_json(run_dir / "result.json", result)
        run.summary.update({"exp2/val": best, "best_epoch": best_epoch, "training_state": "complete"})
        return run_dir
    finally:
        if run is not None:
            run.finish()
        del model, optimizer, scheduler, scaler
        gc.collect(); torch.cuda.empty_cache()


def predict_checkpoint(run_dir, bundle, split="val"):
    """best 예측은 체크포인트·이미지 해시가 같을 때만 재사용한다."""
    run_dir = Path(run_dir)
    manifest = json.loads((run_dir / "config.json").read_text())
    spec, config = manifest["spec"], manifest["config"]
    signature, _ = experiment_identity(bundle, config, spec)
    if signature != manifest["signature"]:
        raise ValueError("분석 코드·데이터·설정이 학습과 다릅니다. 같은 버전을 준비하세요.")
    if split not in ("val", "test"):
        raise ValueError("추론 split은 val 또는 test입니다.")
    dataset = make_dataset(bundle, "val") if split == "val" else TestImages(REPO_ROOT / "data/raw/test_images")
    if split == "test" and len(dataset) != 842:
        raise ValueError("Test는 842장이어야 합니다. test_images.zip을 확인하세요.")
    fingerprint = stable_hash({"checkpoint": file_sha256(run_dir / "weights/best.pt"),
        "images": {row.file_name: image_digest(dataset.image_dir / row.file_name) for row in dataset.frame.itertuples()},
        "signature": signature, "split": split})
    cache = run_dir / f"{split}_predictions.json"
    metadata = run_dir / f"{split}_prediction_identity.json"
    if cache.is_file() and metadata.is_file() and (
            json.loads(metadata.read_text()).get("identity") == fingerprint and
            json.loads(metadata.read_text()).get("predictions_sha256") == file_sha256(cache) and
            (run_dir / f"{split}_manifest.csv").is_file() and
            json.loads(metadata.read_text()).get("manifest_sha256") == file_sha256(run_dir / f"{split}_manifest.csv")):
        return json.loads(cache.read_text()), dataset
    model = build_model(len(bundle["class_map"]), spec, config, load_coco=False).to(_device(config))
    try:
        saved = torch.load(run_dir / "weights/best.pt", map_location="cpu", weights_only=False)
        if saved["signature"] != signature:
            raise ValueError("best 체크포인트가 다른 실험입니다. 실행 폴더를 확인하세요.")
        model.load_state_dict(saved["model"])
        records, speed = collect_predictions(model, dataset, config, bundle["label_to_category"], False)
        if len(records) != len(dataset):
            raise RuntimeError("전체 이미지를 예측하지 못했습니다. 추론을 다시 실행하세요.")
        predictions = predictions_for_team(records)
        write_json(cache, predictions)
        write_frame(run_dir / f"{split}_manifest.csv", dataset.frame)
        write_json(metadata, {"identity": fingerprint, "predictions_sha256": file_sha256(cache), "speed": speed, "images": len(records),
                              "checkpoint_epoch": saved["epoch"],
                              "manifest_sha256": file_sha256(run_dir / f"{split}_manifest.csv")})
        return predictions, dataset
    finally:
        del model; gc.collect(); torch.cuda.empty_cache()


def portability_probe(action="start"):
    """본 실험과 분리해 한 번 갱신·공유 저장·다른 플랫폼 재개를 확인한다."""
    if action not in ("start", "resume"):
        raise ValueError("start 또는 resume을 지정하세요.")
    bundle = prepare_bundle("v2")
    spec = experiment_spec("v2", 5); config = config_for(spec)
    signature, recipe = experiment_identity(bundle, config, spec)
    root = RUN_ROOT / "portability_smoke_v2"
    state = None
    if action == "start":
        if root.exists():
            raise FileExistsError("기존 전환 검사가 있습니다. 덮어쓰지 말고 resume을 사용하세요.")
        root.mkdir(parents=True); (root / "weights").mkdir()
        manifest = {"signature": signature, "run_id": uuid.uuid4().hex[:12], "recipe": recipe,
                    "spec": spec, "config": config, "run_name": root.name}
        write_json(root / "config.json", manifest)
    else:
        if not root.exists() and PLATFORM == "kaggle":
            restore_run_from_drive(root.name)
        manifest = json.loads((root / "config.json").read_text())
        if manifest["signature"] != signature:
            raise ValueError("전환 검사의 코드·데이터·환경 버전이 다릅니다. 같은 버전을 준비하세요.")
        state = torch.load(root / "weights/last.pt", map_location="cpu", weights_only=False)
        if state.get("signature") != signature:
            raise ValueError("전환 검사 체크포인트가 다른 실험입니다.")
    device = _device(config)
    seed_everything(0, config["deterministic"]); generator = torch.Generator().manual_seed(0)
    sample = Subset(make_dataset(bundle, "train"), list(range(16)))
    loader = DataLoader(sample, batch_size=config["batch_size"], shuffle=True, generator=generator,
                        num_workers=0, collate_fn=collate_detection)
    model = build_model(len(bundle["class_map"]), spec, config, load_coco=state is None).to(device)
    optimizer = make_optimizer(model, spec, config)
    scheduler = make_scheduler(optimizer, 1, config)
    scaler = _make_scaler(config, device)
    step = 0
    if state:
        model.load_state_dict(state["model"]); scheduler.load_state_dict(state["scheduler"])
        optimizer.load_state_dict(state["optimizer"]); scaler.load_state_dict(state["scaler"])
        step = int(state["probe_step"])
    import wandb
    run = None
    try:
        run = wandb.init(entity=config["wandb_entity"], project=config["wandb_project"], id=manifest["run_id"],
            name=root.name, tags=["exp2", "smoke", "portability"], resume="must" if state else "never",
            config={"probe": True, "recipe": recipe}, dir=str(root))
        if run is None or run.entity != config["wandb_entity"]:
            raise RuntimeError("팀 W&B run을 확인하지 못했습니다. 계정·팀 권한을 확인하세요.")
        if state:
            _restore_rng(state["rng"], generator)
        before_step = scheduler.last_epoch
        before_weights = model.roi_heads.box_predictor.cls_score.weight.detach().clone()
        batches = next(_accumulation_groups(loader, config["accumulation_steps"]))
        updated = False
        for attempt in range(8):
            model.train()
            losses, updated = _training_step(model, optimizer, scaler, batches, config)
            if updated:
                scheduler.step(); break
        if not updated or scheduler.last_epoch != before_step + 1 or torch.equal(
                before_weights, model.roi_heads.box_predictor.cls_score.weight.detach()):
            raise RuntimeError("전환 검사 갱신이 유효하지 않습니다. AMP·scheduler 상태를 확인하세요.")
        optimizer.zero_grad(set_to_none=True)
        measured = validation_loss(model, Subset(make_dataset(bundle, "val"), [0, 1]), config)
        record = {"signature": signature, "probe_step": step + 1, "model": model.state_dict(),
            "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(), "rng": _rng_state(generator), "run_id": manifest["run_id"]}
        _atomic_save(root / "weights/last.pt", record)
        events_path = root / "environment_history.json"
        events = json.loads(events_path.read_text()) if events_path.exists() else []
        events.append({**runtime_info(), "action": action, "probe_step": step + 1,
                       "scheduler_step_before": before_step, "scheduler_step_after": scheduler.last_epoch,
                       "losses": losses, "validation": measured, **_batch_policy(config)})
        write_json(events_path, events)
        run.log({"smoke/probe_step": step + 1, **losses, **measured}, step=step + 1)
        run.summary.update({"smoke/resume_checked": bool(state), "smoke/platform": PLATFORM,
                            "smoke/shared_folder": RESULT_DRIVE_FOLDER_ID})
    finally:
        if run is not None:
            run.finish()
        del model, optimizer, scheduler, scaler
        gc.collect(); torch.cuda.empty_cache()
    sync_run(root, include_wandb=True)
    print("전환 검사 저장 완료:", action, "갱신", step + 1, "| 본 실험 결과와 분리됨")
    return root


_register_notebook_block("training")


In [ ]:
def detection_errors(records, categories, iou_threshold=0.75, score_threshold=0.30):
    """클래스와 독립적인 IoU 일대일 대응으로 혼동과 미검출을 집계한다.

    Args:
        records (list): validation 공통 결과.
        categories (list): 원래 category ID 목록.
        iou_threshold (float): 대응 최소 IoU.
        score_threshold (float): 진단용 최소 score.

    Returns:
        tuple: 행=정답·열=예측 혼동행렬, 오류 상세 표.
    """
    if not 0 < iou_threshold <= 1 or not 0 <= score_threshold <= 1:
        raise ValueError("진단 IoU·score 범위가 잘못됐습니다. 4번 설정을 확인하세요.")
    categories = [int(c) for c in categories]
    positions = {category: index for index, category in enumerate(categories)}
    background = len(categories)
    matrix = np.zeros((background + 1, background + 1), dtype=int)
    events = []
    for record in records:
        target, pred = record["target"], record["prediction"]
        kept = torch.where(pred["scores"] >= score_threshold)[0]
        boxes, labels, scores = pred["boxes"][kept], pred["labels"][kept], pred["scores"][kept]
        overlaps = box_iou(target["boxes"], boxes).numpy()
        candidates = np.argwhere(overlaps >= iou_threshold)
        candidates = sorted(candidates.tolist(), key=lambda pair: (-overlaps[tuple(pair)], pair))
        used_gt, used_pred = set(), set()

        def append(kind, gt=None, prediction=None, overlap=None):
            events.append({"image_id": int(record["image_id"]), "file_name": record["file_name"],
                           "kind": kind, "gt_index": gt,
                           "pred_index": None if prediction is None else int(kept[prediction]),
                           "true_category": None if gt is None else int(target["labels"][gt]),
                           "pred_category": None if prediction is None else int(labels[prediction]),
                           "score": None if prediction is None else float(scores[prediction]),
                           "iou": overlap})

        for gt, prediction in candidates:
            if gt in used_gt or prediction in used_pred:
                continue
            used_gt.add(gt)
            used_pred.add(prediction)
            truth, estimate = int(target["labels"][gt]), int(labels[prediction])
            matrix[positions[truth], positions[estimate]] += 1
            append("correct" if truth == estimate else "class_confusion", gt, prediction,
                   float(overlaps[gt, prediction]))
        for gt in range(len(target["boxes"])):
            if gt not in used_gt:
                matrix[positions[int(target["labels"][gt])], background] += 1
                possible = [p for p in range(len(boxes)) if int(labels[p]) == int(target["labels"][gt])
                            and 0 < overlaps[gt, p] < iou_threshold]
                best = max(possible, key=lambda p: overlaps[gt, p]) if possible else None
                append("localization_or_missed" if best is not None else "missed", gt, best,
                       None if best is None else float(overlaps[gt, best]))
        for prediction in range(len(boxes)):
            if prediction not in used_pred:
                matrix[background, positions[int(labels[prediction])]] += 1
                duplicate = any(int(target["labels"][gt]) == int(labels[prediction])
                                and overlaps[gt, prediction] >= iou_threshold for gt in used_gt)
                append("duplicate" if duplicate else "extra", prediction=prediction)
    labels = [str(c) for c in categories] + ["unmatched"]
    columns = ["image_id", "file_name", "kind", "gt_index", "pred_index", "true_category",
               "pred_category", "score", "iou"]
    return pd.DataFrame(matrix, index=labels, columns=labels), pd.DataFrame(events, columns=columns)


def plot_confusion(matrix, normalize=True):
    """정답 행 기준 정규화 또는 원시 건수 혼동행렬을 표시한다.

    Args:
        matrix (DataFrame): 행=정답·열=예측. unmatched 행은 추가 예측, 열은 미검출.
        normalize (bool): 정답 행별 비율 표시 여부.

    Returns:
        Figure: 진단 혼동행렬.
    """
    values = matrix.to_numpy(dtype=float)
    if normalize:
        values = np.divide(values, values.sum(axis=1, keepdims=True),
                           out=np.zeros_like(values), where=values.sum(axis=1, keepdims=True) > 0)
    size = max(9, min(20, len(matrix) * 0.28))
    fig, axis = plt.subplots(figsize=(size, size))
    heatmap = axis.imshow(values, cmap="Blues", vmin=0, vmax=1 if normalize else None)
    axis.set(xticks=range(len(matrix)), yticks=range(len(matrix)),
             xticklabels=matrix.columns, yticklabels=matrix.index,
             xlabel="Predicted (unmatched = missed)", ylabel="True (unmatched = extra)",
             title="Detection confusion: row-normalized" if normalize else "Detection confusion: counts")
    axis.tick_params(axis="x", rotation=90, labelsize=7)
    axis.tick_params(axis="y", labelsize=7)
    fig.colorbar(heatmap, ax=axis, fraction=0.04)
    fig.tight_layout()
    return fig


def preview_records(records, events=None, count=6, score_threshold=0.30):
    """정답·예측과 오류 종류를 표시한다. Test에는 정답을 그리지 않는다.

    Args:
        records (list): 공통 예측 결과.
        events (DataFrame | None): validation 오류 표.
        count (int): 최대 이미지 수.
        score_threshold (float): 표시용 점수 하한.

    Returns:
        Figure: 초록 정답·빨강 예측 이미지.
    """
    if events is not None:
        errors = events.loc[events["kind"].ne("correct")].copy()
        errors = errors.sort_values("score", ascending=False, na_position="last")
        ids = list(dict.fromkeys(errors["image_id"].tolist()))[:count]
        chosen = [r for identifier in ids for r in records if r["image_id"] == identifier]
    else:
        chosen = records[:count]
    if not chosen:
        fig, axis = plt.subplots(figsize=(8, 2))
        axis.text(0.5, 0.5, "No error images at the chosen diagnostic thresholds", ha="center")
        axis.axis("off")
        return fig
    fig, axes = plt.subplots(len(chosen), 1, figsize=(9, 6 * len(chosen)), squeeze=False)
    for record, axis in zip(chosen, axes[:, 0]):
        with Image.open(record["image_path"]) as source:
            image = source.convert("RGB")
        if "target" in record:
            target = record["target"]
            image = draw_boxes(image, [xyxy_to_xywh(b.tolist()) for b in target["boxes"]],
                               [f"GT:{int(c)}" for c in target["labels"]], color="lime")
        pred = record["prediction"]
        indices = torch.where(pred["scores"] >= score_threshold)[0]
        indices = indices[torch.argsort(pred["scores"][indices], descending=True)][:20]
        captions = []
        for index in indices:
            caption = f"P:{int(pred['labels'][index])} {float(pred['scores'][index]):.2f}"
            if "target" in record and len(record["target"]["boxes"]):
                overlap = float(box_iou(pred["boxes"][index].reshape(1, 4), record["target"]["boxes"]).max())
                caption += f" I:{overlap:.2f}"
            captions.append(caption)
        image = draw_boxes(image, [xyxy_to_xywh(pred["boxes"][i].tolist()) for i in indices],
                           captions, color="red")
        image.thumbnail((1280, 1280))
        kinds = "prediction only; no ground truth"
        if events is not None:
            kinds = ", ".join(sorted(set(events.loc[events["image_id"].eq(record["image_id"]), "kind"])))
        axis.imshow(image)
        axis.set_title(f"ID {record['image_id']} | {kinds}\ngreen=GT, red=prediction; score>={score_threshold}", fontsize=9)
        axis.axis("off")
    fig.tight_layout()
    return fig


"""공식 val, 두 score의 혼동행렬, IoU별 AP, 정답 없는 test 예측 분석."""
DIAGNOSTIC_SCORES = (0.30, 0.50)
DIAGNOSTIC_IOU = 0.75


def scope_images(gt, scope):
    """전체 val 또는 기존 Kaggle val 이미지만 선택한다."""
    if scope not in ("val", "kaggle_val47"):
        raise ValueError("분석 범위는 val 또는 kaggle_val47입니다.")
    return [image for image in gt["images"]
            if scope == "val" or image.get("source") == "kaggle"]


def prediction_records(predictions, dataset):
    """예측이 없는 이미지를 포함해 tensor 표시 형식으로 연결한다."""
    by_name = defaultdict(list)
    for pred in predictions:
        by_name[pred["file_name"]].append(pred)
    if set(by_name) - set(dataset.frame.file_name):
        raise ValueError("추론 대상에 없는 이미지 이름입니다. 예측 파일·manifest를 확인하세요.")
    records = []
    for row in dataset.frame.itertuples():
        values = by_name[row.file_name]
        boxes = torch.tensor([p["bbox"] for p in values], dtype=torch.float32).reshape(-1, 4)
        records.append({"image_id": int(row.image_id), "file_name": row.file_name,
                        "image_path": str(dataset.image_dir / row.file_name),
                        "prediction": {"boxes": box_convert(boxes, "xywh", "xyxy"),
                            "labels": torch.tensor([p["category_id"] for p in values], dtype=torch.int64),
                            "scores": torch.tensor([p["score"] for p in values], dtype=torch.float32)}})
    return records


def diagnostic_records(predictions, bundle, scope="val"):
    """팀 무시 규칙 적용 후 원래 K-코드의 정답을 붙인다."""
    gt = bundle["gt"]
    images = scope_images(gt, scope)
    names = {row["file_name"] for row in images}
    competition = {row["id"] for row in gt["categories"] if row["competition"]}
    annotations = defaultdict(list)
    for ann in gt["annotations"]:
        if ann["category_id"] in competition:
            annotations[ann["image_id"]].append(ann)
    kept = drop_ignored(predictions, gt)
    records = prediction_records(kept, make_dataset(bundle, "val"))
    image_ids = {row["file_name"]: row["id"] for row in images}
    result = []
    for record in records:
        if record["file_name"] not in names:
            continue
        values = annotations[image_ids[record["file_name"]]]
        boxes = torch.tensor([ann["bbox"] for ann in values], dtype=torch.float32).reshape(-1, 4)
        record["target"] = {"boxes": box_convert(boxes, "xywh", "xyxy"),
                            "labels": torch.tensor([a["category_id"] for a in values], dtype=torch.int64)}
        result.append(record)
    return result


def diagnostic_summary(matrix):
    """오분류는 해당 클래스 FP·FN에 반영하고 분모 0은 None으로 기록한다."""
    values = matrix.to_numpy(dtype=int)
    rows = []
    def ratio(numerator, denominator):
        return float(numerator / denominator) if denominator else None
    for index, category in enumerate(matrix.index[:-1]):
        tp = int(values[index, index])
        fp, fn = int(values[:, index].sum() - tp), int(values[index, :].sum() - tp)
        rows.append({"category_id": int(category), "TP": tp, "FP": fp, "FN": fn,
                     "GT_objects": tp + fn, "precision": ratio(tp, tp + fp),
                     "recall": ratio(tp, tp + fn), "F1": ratio(2 * tp, 2 * tp + fp + fn)})
    frame = pd.DataFrame(rows)
    tp, fp, fn = (int(frame[column].sum()) for column in ("TP", "FP", "FN"))
    return frame, {"average": "micro", "TP": tp, "FP": fp, "FN": fn,
                   "precision": ratio(tp, tp + fp), "recall": ratio(tp, tp + fn),
                   "F1": ratio(2 * tp, 2 * tp + fp + fn),
                   "zero_denominator": "null / N/A", "iou": DIAGNOSTIC_IOU,
                   "matching": "class-independent greedy descending IoU; one-to-one"}


def class_iou_ap(predictions, gt, scope="val"):
    """공식 평가와 같은 COCO precision에서 클래스·IoU별 AP를 얻는다."""
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    images = scope_images(gt, scope)
    ids = {image["id"] for image in images}
    names = {image["file_name"]: image["id"] for image in images}
    competition = sorted(c["id"] for c in gt["categories"] if c["competition"])
    supports = Counter(ann["category_id"] for ann in gt["annotations"] if ann["image_id"] in ids)
    present = [category for category in competition if supports[category] > 0]
    kept = [pred for pred in drop_ignored(predictions, gt) if pred["file_name"] in names]
    results = [{"image_id": names[p["file_name"]], "category_id": p["category_id"],
                "bbox": list(p["bbox"]), "score": p["score"]} for p in kept]
    values_of = {category: ([0.0] * 5 if supports[category] else [None] * 5)
                 for category in competition}
    if results and present:
        with contextlib.redirect_stdout(io.StringIO()):
            coco = COCO(); coco.dataset = copy.deepcopy(gt); coco.createIndex()
            evaluator = COCOeval(coco, coco.loadRes(results), "bbox")
            evaluator.params.iouThrs = np.linspace(0.75, 0.95, 5)
            evaluator.params.imgIds = sorted(ids); evaluator.params.catIds = present
            evaluator.evaluate(); evaluator.accumulate()
        precision = evaluator.eval["precision"][:, :, :, 0, -1]
        for index, category in enumerate(evaluator.params.catIds):
            values_of[int(category)] = [float(row[row > -1].mean()) if (row > -1).any() else None
                                        for row in precision[:, :, index]]
    metadata = {c["id"]: c for c in gt["categories"]}
    rows = []
    for category, values in values_of.items():
        valid = [value for value in values if value is not None]
        rows.append({"scope": scope, "category_id": category,
            "k_code": str(metadata[category].get("k_code", category)),
            "name": str(metadata[category].get("product_name", metadata[category].get("name", category))),
            "GT_objects": supports[category],
            **dict(zip(("AP75", "AP80", "AP85", "AP90", "AP95"), values)),
            "AP75_95": float(np.mean(valid)) if valid else None,
            "AP75_minus_AP95": values[0] - values[-1] if valid else None})
    return pd.DataFrame(rows)


def plot_iou_ap(frame, run_name, directory):
    """AP 산점도와 평균 AP 상·하위 5개 클래스의 IoU 곡선을 저장한다."""
    valid = frame.loc[frame.GT_objects.gt(0)].sort_values("AP75_95", ascending=False)
    fig, axis = plt.subplots(figsize=(7, 6))
    points = axis.scatter(valid.AP75, valid.AP95, c=valid.GT_objects, cmap="viridis", alpha=0.8)
    axis.plot([0, 1], [0, 1], "k--", label="AP75 = AP95")
    axis.set(xlim=(-.02, 1.02), ylim=(-.02, 1.02), xlabel="AP75", ylabel="AP95",
             title=f"{run_name} | {frame.scope.iloc[0]} | GT > 0")
    axis.legend(); fig.colorbar(points, ax=axis, label="GT objects")
    save_figure(directory / "ap75_ap95_scatter.png", fig)
    fig, axis = plt.subplots(figsize=(10, 6))
    top = valid.head(min(5, len(valid)))
    bottom = valid.loc[~valid.category_id.isin(top.category_id)].tail(5)
    for label, group, style in (("high mean AP", top, "-"), ("low mean AP", bottom, "--")):
        for row in group.itertuples():
            axis.plot([.75, .80, .85, .90, .95], [getattr(row, f"AP{i}") for i in (75,80,85,90,95)],
                      style, marker="o", alpha=.7,
                      label=f"{label}: {row.category_id} (GT={row.GT_objects})")
    axis.set(xlabel="IoU threshold", ylabel="AP", ylim=(-.02, 1.02),
             title=f"{run_name} | {frame.scope.iloc[0]} | groups ranked by mean AP75:95")
    axis.legend(fontsize=8, loc="center left", bbox_to_anchor=(1, .5))
    save_figure(directory / "class_iou_ap_curves.png", fig)


def analyze_validation(predictions, bundle, run_dir, plots=True):
    """같은 best 예측으로 전체 val·47장 진단과 AP 차이를 저장한다."""
    run_dir = Path(run_dir)
    scores = evaluate_val(predictions, bundle["gt"])
    write_json(run_dir / "scores.json", scores)
    competition = sorted(c["id"] for c in bundle["gt"]["categories"] if c["competition"])
    for scope in ("val", "kaggle_val47"):
        records = diagnostic_records(predictions, bundle, scope)
        directory = run_dir / "analysis" / scope
        directory.mkdir(parents=True, exist_ok=True)
        ap = class_iou_ap(predictions, bundle["gt"], scope)
        ap.insert(0, "run", run_dir.name)
        write_frame(directory / "class_iou_ap.csv", ap)
        write_frame(directory / "low_ap_classes.csv", ap.loc[ap.GT_objects.gt(0)].sort_values("AP75_95"))
        if scope == "val":
            write_frame(run_dir / "analysis/class_iou_ap.csv", ap)
        if plots:
            plot_iou_ap(ap, run_dir.name, directory)
            if scope == "val":
                for name in ("ap75_ap95_scatter.png", "class_iou_ap_curves.png"):
                    shutil.copy2(directory / name, run_dir / "analysis" / name)
            large_gap = set(ap.loc[ap.GT_objects.gt(0)].sort_values(
                ["AP75_minus_AP95", "AP75"], ascending=False).head(5).category_id)
            chosen = [r for r in records if any(int(c) in large_gap for c in r["target"]["labels"])]
            save_figure(directory / "large_ap_gap_examples.png", preview_records(chosen, count=3))
        for score in DIAGNOSTIC_SCORES:
            output = directory / f"score_{score:.2f}"
            matrix, events = detection_errors(records, competition, DIAGNOSTIC_IOU, score)
            per_class, summary = diagnostic_summary(matrix)
            summary.update(score=score, scope=scope, images=len(records),
                           ignored_rule="team drop_ignored, noncompetition GT excluded")
            write_json(output / "summary.json", summary)
            write_frame(output / "per_class.csv", per_class)
            write_frame(output / "errors.csv", events)
            write_frame(output / "confusion_counts.csv", matrix.reset_index(names="true_category"))
            denominators = matrix.sum(axis=1).replace(0, np.nan)
            normalized = matrix.div(denominators, axis=0)
            write_frame(output / "confusion_row_normalized.csv", normalized.reset_index(names="true_category"))
            if plots:
                for normalize, name in ((False, "confusion_counts.png"), (True, "confusion_row_normalized.png")):
                    fig = plot_confusion(matrix, normalize)
                    fig.suptitle(f"{run_dir.name} | {scope} | score >= {score} | IoU >= {DIAGNOSTIC_IOU}")
                    save_figure(output / name, fig)
                save_figure(output / "error_examples.png", preview_records(records, events, count=3, score_threshold=score))
    write_json(run_dir / "analysis/metric_definitions.json", {
        "official_AP": "score>=0.001; IoU75:95; team ignore rules; maxDets=100; all area",
        "diagnostics": "scores .30/.50; IoU .75; micro + per class; zero denominator null",
        "kaggle_val47": "validation images, not leaderboard",
        "test_only": "18 classes evaluated on val, not test images",
        "AP_group_selection": "top5 and disjoint bottom5 by mean AP75:95 among GT>0",
        "AP_gap_warning": "Small gap with two low APs is not evidence of good localization."})
    return scores


def analyze_test(predictions, dataset, bundle, run_dir, plots=True, known_training_counts=True):
    """Test는 정답 없는 예측 분포·겹침만 기록하고 정확도 지표를 만들지 않는다."""
    directory = Path(run_dir) / "analysis/test"
    competition = competition_from_classes(bundle["root"] / "classes.csv")
    records = prediction_records(predictions, dataset)
    train_counts = Counter(a["category_id"] for a in make_dataset(bundle, "train").doc["annotations"])
    val_counts = Counter(a["category_id"] for a in bundle["gt"]["annotations"])
    if OTHERS_CATEGORY_ID in bundle["category_to_label"]:
        val_counts[OTHERS_CATEGORY_ID] = sum(count for category, count in val_counts.items()
                                             if category not in competition and category != OTHERS_CATEGORY_ID)
    columns = ["file_name", "category_id", "score"]
    all_scores = pd.DataFrame([{k: pred[k] for k in columns} for pred in predictions], columns=columns)
    write_frame(directory / "scores.csv", all_scores)
    classes = sorted(competition | set(bundle["category_to_label"]) | {p["category_id"] for p in predictions})
    for score in DIAGNOSTIC_SCORES:
        output = directory / f"score_{score:.2f}"
        kept = [p for p in predictions if p["score"] >= score]
        counts = Counter(p["category_id"] for p in kept)
        per_class = pd.DataFrame([{"category_id": category,
            "group": "competition" if category in competition else (
                "others" if category == OTHERS_CATEGORY_ID else "noncompetition"),
            "train_GT_objects": train_counts[category] if known_training_counts else None,
            "val_GT_objects": val_counts[category],
            "test_predictions": counts[category]} for category in classes])
        write_frame(output / "class_counts.csv", per_class)
        per_image = []
        overlaps = []
        for record in records:
            pred = record["prediction"]
            indices = torch.where(pred["scores"] >= score)[0]
            labels = pred["labels"][indices]
            per_image.append({"file_name": record["file_name"], "detections_all": len(indices),
                "detections_competition": sum(int(c) in competition for c in labels),
                "detections_others_noncompetition": sum(int(c) not in competition for c in labels)})
            ious = box_iou(pred["boxes"][indices], pred["boxes"][indices])
            for i, j in torch.nonzero(torch.triu(ious >= .5, diagonal=1)).tolist():
                overlaps.append({"file_name": record["file_name"], "iou": float(ious[i,j]),
                    "category_a": int(labels[i]), "category_b": int(labels[j]),
                    "score_a": float(pred["scores"][indices[i]]), "score_b": float(pred["scores"][indices[j]]),
                    "different_class": bool(labels[i] != labels[j]),
                    "both_competition": bool(int(labels[i]) in competition and int(labels[j]) in competition)})
        write_frame(output / "image_counts.csv", pd.DataFrame(per_image))
        overlap_frame = pd.DataFrame(overlaps, columns=["file_name", "iou", "category_a", "category_b",
            "score_a", "score_b", "different_class", "both_competition"])
        write_frame(output / "overlapping_predictions.csv", overlap_frame)
        write_json(output / "summary.json", {"images": len(records), "predictions": len(kept),
            "competition_predictions": sum(p["category_id"] in competition for p in kept),
            "zero_detection_images": sum(row["detections_all"] == 0 for row in per_image),
            "score": score, "overlap_iou": .5, "ground_truth": "unavailable; prediction distribution only"})
        if plots:
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].hist([p["score"] for p in kept], bins=np.linspace(score, 1, 21))
            axes[0].set(xlabel="Prediction score", ylabel="Count", title=f"test | score >= {score}")
            axes[1].hist([row["detections_competition"] for row in per_image], bins=20)
            axes[1].set(xlabel="Competition predictions per image", ylabel="Images", title=f"all {len(records)} images")
            fig.suptitle(Path(run_dir).name + " | predictions, no GT")
            save_figure(output / "prediction_distributions.png", fig)
            conflict_names = set(overlap_frame.loc[overlap_frame.different_class.eq(True), "file_name"])
            conflicts = [r for r in records if r["file_name"] in conflict_names]
            save_figure(output / "overlap_examples.png", preview_records(conflicts, count=3, score_threshold=score))
            low = sorted(records, key=lambda r: min((float(v) for v in r["prediction"]["scores"] if v >= score), default=2))
            save_figure(output / "low_score_examples.png", preview_records(low, count=3, score_threshold=score))
    return {"images": len(records), "predictions": len(predictions)}


def plot_training(run_dir):
    """학습 loss와 세 val 점수의 곡선을 저장한다."""
    history = pd.read_csv(Path(run_dir) / "metrics.csv")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.epoch, history.train_loss, label="train loss")
    axes[0].plot(history.epoch, history.val_loss, label="val loss")
    for column in ("val", "kaggle", "test_only"):
        axes[1].plot(history.epoch, history[column], label=column + " (val)")
    for axis in axes:
        axis.set_xlabel("Epoch"); axis.legend()
    axes[0].set_ylabel("Loss"); axes[1].set_ylabel("mAP75:95")
    fig.suptitle(Path(run_dir).name)
    save_figure(Path(run_dir) / "analysis/training_curves.png", fig)


def record_run_summary(run_dir, save=True):
    """완료·미완료를 구분하는 작은 비교표를 공유 저장과 팀 경로에 쓴다."""
    rows = []
    for epochs in EPOCH_BUDGETS:
        for variant in VARIANTS:
            name = experiment_spec(variant, epochs)["run_name"]
            root = RUN_ROOT / name
            result = json.loads((root / "result.json").read_text()) if (root / "result.json").exists() else {}
            manifest = json.loads((root / "config.json").read_text()) if (root / "config.json").exists() else {}
            scores = json.loads((root / "scores.json").read_text()) if (root / "scores.json").exists() else {}
            state = "analysis_complete" if (root / "analysis_complete.json").exists() else (
                result.get("state", "interrupted" if manifest else "not_started"))
            rows.append({"run": name, "variant": variant, "epochs": epochs, "seed": 0,
                "state": state, "completed_epochs": result.get("last_epoch"), "best_epoch": result.get("best_epoch"),
                **{key: scores.get(key) for key in ("val", "kaggle", "test_only")},
                "training_seconds": result.get("training_seconds"), "data_identity": manifest.get("recipe", {}).get("data_identity"),
                "signature": manifest.get("signature"),
                "wandb_url": f"https://wandb.ai/{DEFAULT_CONFIG['wandb_entity']}/{DEFAULT_CONFIG['wandb_project']}/runs/{manifest['run_id']}" if manifest else None,
                "best_path": str(root / "weights/best.pt") if manifest else None})
    frame = pd.DataFrame(rows)
    if save:
        write_frame(RUN_ROOT / "runs.csv", frame)
        write_frame(REPO_ROOT / "notebooks/modeling/02-cj-faster-rcnn-exp2/runs.csv", frame)
        if PLATFORM == "kaggle":
            drive_upload(RUN_ROOT / "runs.csv", RESULT_DRIVE_FOLDER_ID)
    return frame


def finalize_run(run_dir, bundle, plots=True):
    """best의 val·test 자료를 저장하고 같은 W&B run에 연결한다."""
    run_dir = Path(run_dir)
    manifest = json.loads((run_dir / "config.json").read_text())
    predictions, _ = predict_checkpoint(run_dir, bundle, "val")
    scores = analyze_validation(predictions, bundle, run_dir, plots)
    test_predictions, dataset = predict_checkpoint(run_dir, bundle, "test")
    test_summary = analyze_test(test_predictions, dataset, bundle, run_dir, plots)
    if plots:
        plot_training(run_dir)
    import wandb
    run = wandb.init(entity=DEFAULT_CONFIG["wandb_entity"], project=DEFAULT_CONFIG["wandb_project"],
        id=manifest["run_id"], resume="must", dir=str(run_dir))
    try:
        run.summary.update({**{f"exp2/{k}": scores[k] for k in ("val", "kaggle", "test_only")},
            "exp2/state": "analysis_complete", "exp2/shared_result": str(run_dir),
            "exp2/drive_folder": "https://drive.google.com/drive/folders/" + RESULT_DRIVE_FOLDER_ID,
            "exp2/test_images": test_summary["images"]})
        run.log({"analysis/class_iou_ap": wandb.Table(dataframe=pd.read_csv(run_dir / "analysis/class_iou_ap.csv"))})
        if plots:
            run.log({"analysis/training_curves": wandb.Image(str(run_dir / "analysis/training_curves.png")),
                     "analysis/AP75_AP95": wandb.Image(str(run_dir / "analysis/ap75_ap95_scatter.png"))})
    finally:
        run.finish()
    write_json(run_dir / "analysis_complete.json", {"signature": manifest["signature"], "plots": plots,
        "checkpoint_sha256": file_sha256(run_dir / "weights/best.pt"), "scores": scores,
        "test": test_summary, "time_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})
    sync_run(run_dir, include_wandb=True)
    return record_run_summary(run_dir)


def run_selected(selection, plots=False):
    """9개 계획 중 선택한 순서대로 독립 학습하고 기본 분석까지 보존한다."""
    plan = experiment_plan()
    if not selection or len(set(selection)) != len(selection) or set(selection) - set(plan.run_name):
        raise ValueError("실행할 run 이름을 중복 없이 선택하세요.")
    for name in selection:
        spec = plan.loc[plan.run_name.eq(name), list(experiment_spec("v2", 5))].iloc[0].to_dict()
        spec["epochs"] = int(spec["epochs"]); spec["seed"] = int(spec["seed"])
        bundle = prepare_bundle(spec["variant"])
        run_dir = run_experiment(bundle, spec, resume=name in RESUME_RUN_NAMES)
        marker = run_dir / "analysis_complete.json"
        completed = json.loads(marker.read_text()) if marker.exists() else {}
        if completed.get("checkpoint_sha256") == file_sha256(run_dir / "weights/best.pt") and (
                not plots or completed.get("plots")) and all((run_dir / path).is_file() for path in (
                    "scores.json", "test_predictions.json", "test_manifest.csv", "analysis/class_iou_ap.csv")):
            print("보존 완료 run 건너뜀:", name)
            sync_run(run_dir, include_wandb=True)
            continue
        finalize_run(run_dir, bundle, plots)
    return record_run_summary(RUN_ROOT)




def reevaluate_first_run(first_run_dir, bundle, plots=True):
    """기존 run의 매핑·설정을 확인한 뒤 공통 val·test를 다시 예측한다."""
    first_run_dir = Path(first_run_dir)
    manifest = json.loads((first_run_dir / "config.json").read_text())
    old_config = manifest["recipe"]["config"]
    mapping = pd.read_csv(first_run_dir / "class_map.csv")
    if not {"model_label", "category_id"}.issubset(mapping):
        raise ValueError("1차 class_map 형식을 확인하세요. 모델 label·원래 category ID가 필요합니다.")
    if sorted(mapping.model_label.tolist()) != list(range(1, len(mapping) + 1)):
        raise ValueError("1차 모델 label이 연속적이지 않습니다. 저장된 매핑을 확인하세요.")
    for key in ("image_size", "score_threshold", "nms_iou", "max_detections", "normalization"):
        if old_config.get(key) != DEFAULT_CONFIG[key]:
            raise ValueError(f"1차 추론 설정 {key}가 다릅니다. 임의 변경 없이 비교 조건을 먼저 확인하세요.")
    checkpoint = first_run_dir / "best.pt"
    if not checkpoint.is_file():
        checkpoint = first_run_dir / "weights/best.pt"
    if not checkpoint.is_file():
        raise FileNotFoundError("1차 best.pt가 없습니다. 실제 저장 위치를 지정하세요.")
    reverse = dict(zip(mapping.model_label.astype(int), mapping.category_id.astype(int)))
    output = RUN_ROOT / "first_round_reevaluation" / file_sha256(checkpoint)[:12]
    output.mkdir(parents=True, exist_ok=True)
    spec = experiment_spec("v2", 5); config = config_for(spec)
    model = None
    try:
        for split, dataset in (("val", make_dataset(bundle, "val")),
                               ("test", TestImages(REPO_ROOT / "data/raw/test_images"))):
            if split == "test" and len(dataset) != 842:
                raise ValueError("1차 비교도 동일한 test 842장을 사용해야 합니다.")
            identity = stable_hash({"checkpoint": file_sha256(checkpoint), "config": old_config,
                "mapping": {str(k): v for k, v in reverse.items()}, "evaluation_code": code_identity(),
                "images": {row.file_name: image_digest(dataset.image_dir / row.file_name) for row in dataset.frame.itertuples()},
                "split": split, "gt": stable_hash(bundle["gt"])})
            cache, meta_path = output / f"{split}_predictions.json", output / f"{split}_cache_identity.json"
            meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
            if cache.is_file() and meta.get("identity") == identity and meta.get("sha256") == file_sha256(cache):
                predictions = json.loads(cache.read_text())
            else:
                if model is None:
                    model = build_model(len(mapping), spec, config, load_coco=False).to(_device(config))
                    saved = torch.load(checkpoint, map_location="cpu", weights_only=False)
                    if saved.get("signature") != manifest.get("signature"):
                        raise ValueError("1차 체크포인트와 config가 다릅니다. 같은 run의 파일을 지정하세요.")
                    model.load_state_dict(saved["model"])
                records, speed = collect_predictions(model, dataset, config, reverse, with_targets=False)
                predictions = predictions_for_team(records)
                write_json(cache, predictions)
                write_json(meta_path, {"identity": identity, "sha256": file_sha256(cache), "speed": speed})
            write_frame(output / f"{split}_manifest.csv", dataset.frame)
            if split == "val":
                scores = analyze_validation(predictions, bundle, output, plots)
            else:
                first_bundle = {**bundle, "category_to_label": {v: k for k, v in reverse.items()}}
                summary = analyze_test(predictions, dataset, first_bundle, output, plots, known_training_counts=False)
        write_json(output / "comparison_identity.json", {"first_checkpoint_sha256": file_sha256(checkpoint),
            "first_signature": manifest["signature"], "first_config": old_config,
            "evaluation_data_identity": bundle["identity"], "evaluation_code": code_identity(),
            "first_train_frequencies": "not loaded; N/A", "scores": scores, "test": summary})
        print("1차 공통 규칙 재평가 저장:", output)
        return scores
    finally:
        del model; gc.collect(); torch.cuda.empty_cache()


_register_notebook_block("analysis")


## 4. 공유 Drive 연결

Colab은 `MOUNT_COLAB_DRIVE=True`로 연결합니다. 1번 셀 `DRIVE_ROOT` 아래에 zip 4개를 둔 `data/`가 있는지 확인합니다. 결과 폴더 `rw_runs/`는 없으면 만듭니다. `data/`가 없으면 `DRIVE_ROOT`가 틀린 것이므로 폴더를 만들지 않고 멈춥니다.

Kaggle은 `CONNECT_KAGGLE_DRIVE=True`로 연결합니다. 공유 폴더에 접근 가능한 Google Drive API access token을 **숨김 입력창**에 넣습니다. 토큰을 코드·출력·W&B에 적지 마세요. 만료 시 `connect_drive()`로 갱신합니다. Kaggle의 기본 `/kaggle/working`은 임시 공간이므로 매 epoch 공유 Drive 업로드·체크섬 확인까지 완료되어야 보존된 것입니다. 토큰 발급과 실제 권한은 예비 환경 설정 단계에서 함께 확인합니다.

In [ ]:
MOUNT_COLAB_DRIVE = True
CONNECT_KAGGLE_DRIVE = False
if MOUNT_COLAB_DRIVE and PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    if not ZIP_ROOT.is_dir():
        raise FileNotFoundError(f"zip 폴더가 없습니다: {ZIP_ROOT}. 1번 셀의 DRIVE_ROOT를 Drive의 실제 경로로 고치세요.")
    RUN_ROOT.mkdir(exist_ok=True)  # DRIVE_ROOT가 맞을 때만 rw_runs를 만든다
if CONNECT_KAGGLE_DRIVE and PLATFORM == "kaggle":
    connect_drive()
    print("공유 데이터 항목:", [row["name"] for row in drive_children(DATA_DRIVE_FOLDER_ID)])
    drive_children(RESULT_DRIVE_FOLDER_ID)  # 읽기 확인. 쓰기는 전환 검사에서 확인한다.


## 5. 데이터 ZIP 확인·전개와 고정 목록 검사

`PREPARE_DATA=True`일 때만 ZIP 4개를 처리합니다. 기존 파일은 ZIP CRC·크기가 같으면 재사용하며, 다르면 덮어쓰지 않고 중단합니다. 팀에서 제공한 정제 ZIP을 사용하고 예전 라벨 수정표를 다시 적용하지 않습니다. 완료 후 `make_dataset_lists.py --check`의 두 CSV가 모두 **같음**이어야 합니다. 고정 목록은 다시 생성하지 않습니다.

In [ ]:
PREPARE_DATA = True
if PREPARE_DATA:
    if PLATFORM == "local":
        raise RuntimeError("로컬 원본 데이터는 여기서 바꾸지 않습니다. 클라우드 환경에서 실행하세요.")
    for archive_name in ("kaggle.zip", "aihub_annotations.zip", "aihub_images.zip", "test_images.zip"):
        archive = download_team_zip(archive_name) if PLATFORM == "kaggle" else ZIP_ROOT / archive_name
        if not archive.is_file():
            raise FileNotFoundError(f"ZIP이 없습니다: {archive}. 공유 폴더 바로가기를 확인하세요.")
        with zipfile.ZipFile(archive) as source:
            for member in source.infolist():
                parts = Path(member.filename).parts
                if "__MACOSX" in parts or ".DS_Store" in parts or any(p.startswith("._") for p in parts):
                    continue
                destination = (REPO_ROOT / member.filename).resolve()
                if not destination.is_relative_to((REPO_ROOT / "data").resolve()):
                    raise ValueError(f"data 밖의 ZIP 항목입니다: {member.filename}. 팀 ZIP을 확인하세요.")
                if stat.S_ISLNK(member.external_attr >> 16):
                    raise ValueError("ZIP에 심볼릭 링크가 있습니다. 팀 ZIP 구성을 확인하세요.")
                if member.is_dir():
                    destination.mkdir(parents=True, exist_ok=True)
                    continue
                if destination.exists():
                    import zlib
                    crc = 0
                    with destination.open("rb") as stream:
                        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b""):
                            crc = zlib.crc32(chunk, crc)
                    if destination.stat().st_size != member.file_size or crc != member.CRC:
                        raise ValueError(f"기존 파일과 ZIP 내용이 다릅니다: {member.filename}. 덮어쓰지 말고 확인하세요.")
                    continue
                destination.parent.mkdir(parents=True, exist_ok=True)
                temporary = destination.with_name(destination.name + ".part")
                with source.open(member) as input_stream, temporary.open("wb") as output_stream:
                    shutil.copyfileobj(input_stream, output_stream)
                os.replace(temporary, destination)
        print("ZIP 확인·전개 완료:", archive_name)
    environment = {**os.environ, "PYTHONPATH": str(REPO_ROOT / "src"), "PYTHONDONTWRITEBYTECODE": "1"}
    checked = subprocess.run([sys.executable, str(REPO_ROOT / "scripts/make_dataset_lists.py"), "--check"],
                             cwd=REPO_ROOT, env=environment, capture_output=True, text=True)
    print(checked.stdout)
    checked.check_returncode()
    if "images.csv: 같음" not in checked.stdout or "classes.csv: 같음" not in checked.stdout:
        raise RuntimeError("고정 목록 일치 여부를 확인하지 못했습니다. --check 출력을 확인하세요.")


## 6. v2·v3·v4 학습 폴더 생성

`BUILD_DATASETS=True`로 팀 `build_dataset.py --format coco`를 사용합니다. 폴더가 이미 있으면 목록 해시·개수·매핑·공통 val·이미지 내용을 검사해 재사용합니다. train/val 이름 또는 같은 이미지 내용이 겹치면 중단합니다.

| 방법 | train 이미지 | 전경 클래스 | 배경 포함 |
| --- | ---: | ---: | ---: |
| v2: 최소 200개 목표 | 7,129 | 118 | 119 |
| v3: 대회 클래스 포함 사진 | 8,974 | 96 | 97 |
| v4: 비대회 알약 → others | 11,206 | 75 | 76 |

공식 val은 방법 공통 **1,000장·3,790객체**입니다. 학습·val loss는 방법별 COCO 라벨을 사용하고, 점수·혼동행렬은 원래 K-코드의 `eval/val_gt.json`을 사용합니다. `idx+1`이 모델 label이며 0은 배경입니다. 처음 이미지 해시 계산에는 시간이 걸릴 수 있습니다.

In [ ]:
BUILD_DATASETS = True
if BUILD_DATASETS:
    environment = {**os.environ, "PYTHONPATH": str(REPO_ROOT / "src"), "PYTHONDONTWRITEBYTECODE": "1"}
    for variant in VARIANTS:
        destination = DATASET_ROOT / variant
        if not destination.exists() or not any(destination.iterdir()):
            subprocess.run([sys.executable, str(REPO_ROOT / "scripts/build_dataset.py"),
                "--variant", variant, "--format", "coco", "--out", str(destination)],
                cwd=REPO_ROOT, env=environment, check=True)
        else:
            print("기존 생성 폴더를 검증해 재사용:", destination)
        prepare_bundle(variant)


## 7. 실험 선택과 데이터 미리 보기

`SELECTED_RUNS`로 실행 순서를 선택합니다. 기본값은 9개지만 첫 run만 선택해서 시간을 확인할 수 있습니다. 완료 run은 다시 학습하지 않습니다. 중단 run은 `RESUME_RUN_NAMES`에 명시해야 재개합니다. 실효 batch 16을 포함한 학습 설정은 고정입니다. 실제 batch는 1번 셀 `MICRO_BATCH`로 정하고 자동으로 바꾸지 않으며, 같은 run은 같은 값으로만 재개합니다.

In [ ]:
# 기본 선택: VARIANTS × EPOCH_BUDGETS 전부. 첫 실행에서는 SELECTED_RUNS를 ["v3_frcnn_e5_s0"]처럼 줄여도 된다.
SELECTED_RUNS = [experiment_spec(v, e)["run_name"] for e in EPOCH_BUDGETS for v in VARIANTS]
# 무거운 그림은 학습 후 별도 분석 셀에서 생성한다. 숫자·val/test 캐시는 매 run 보존한다.
MAKE_PLOTS_DURING_QUEUE = False
print(f"고정: 실효 batch {EFFECTIVE_BATCH} = 실제 {MICRO_BATCH} × 누적 {EFFECTIVE_BATCH // MICRO_BATCH} | seed 0 | "
      f"각 예산 독립 학습 | 로더 worker {LOADER_WORKERS}")
display(experiment_plan())


In [ ]:
INSPECT_DATA = False
if INSPECT_DATA:
    for variant in VARIANTS:
        show_data(prepare_bundle(variant))


## 8. 학습 전 짧은 검사와 Colab → Kaggle 전환 검사

먼저 Colab에서 `PROBE_ACTION="start"`로 실제 **batch·누적(`MICRO_BATCH`)·AdamW·AMP** 1회 갱신, 검증 FP32 loss, 전체 상태 저장, W&B·공유 보존을 확인합니다. 다음으로 Colab 실행을 멈춘 상태에서 Kaggle에서 **같은 노트북·패키지·데이터**를 준비하고 `PROBE_ACTION="resume"`을 실행합니다. 모델·optimizer·scheduler·scaler·난수·동일 W&B ID를 복원해 다음 갱신과 공유 저장을 확인합니다.

검사는 `portability_smoke_v2`에 분리 저장합니다. 본 실험의 max_epochs를 1로 바꾸거나 검사 가중치를 재사용하지 않습니다. 이 검사까지 실제로 통과해야 Kaggle 예비 환경 준비 완료입니다. 본 학습도 방법별로 별도의 1회 갱신 검사를 먼저 수행합니다.

In [ ]:
PROBE_ACTION = None  # Colab: "start" → 저장 확인 → Kaggle: "resume"
if PROBE_ACTION is not None:
    import wandb
    wandb.login()  # 키는 노트북 소스·출력에 쓰지 않고 로그인 창에 입력한다.
    portability_probe(PROBE_ACTION)
else:
    print("전환 검사는 아직 실행하지 않았습니다. 본 실험 전 Colab start와 Kaggle resume을 확인하세요.")


## 9. 본 학습 시작

**이 셀이 장시간 학습을 시작합니다.** 환경·전환 검사와 선택한 run을 함께 확인한 뒤 `START_TRAINING=True`로 바꿉니다. 함수·분석 셀만 실행해서 학습이 시작되지는 않습니다.

학습 중에는 100 step마다 처리 속도(장/초)를 출력합니다. 매 epoch train loss → val loss(FP32) → val 1회 추론 → 공식 세 점수 → best/last 저장 → 공유 보존 → W&B 기록 순서입니다. 매 run 종료 후 best의 val·test 예측, 두 score의 진단 표, AP 표를 저장합니다. 큰 그림은 별도 분석 셀에서 캐시로 생성해도 됩니다.

중단 시 완료된 마지막 epoch 이후부터 재개하며 미완료 epoch는 다시 실행합니다. 최근 정상 last/best도 보존합니다. 속도를 위해 결정적 연산을 껐으므로 같은 GPU에서 다시 돌리거나 재개해도 수치가 완전히 같지는 않습니다. W&B 숫자 기록·공유 업로드에 실패하면 숨기지 않고 멈춥니다.

In [ ]:
START_TRAINING = True  # 환경·데이터·전환 검사 후 함께 확인하고 True로 바꾼다.
if START_TRAINING:
    display(experiment_plan().loc[lambda frame: frame.run_name.isin(SELECTED_RUNS)])
    import wandb
    wandb.login()
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    run_selected(SELECTED_RUNS, plots=MAKE_PLOTS_DURING_QUEUE)
else:
    print("학습은 시작하지 않았습니다. 이 셀의 START_TRAINING=True가 본 학습 시작입니다.")


## 10. 중단된 run 재개

Colab은 같은 Drive 경로를 사용합니다. Kaggle 새 세션은 아래 `RESTORE_RUN_NAMES`로 공유 run을 먼저 복원합니다. 복원이 끝난 뒤 **7번의 `RESUME_RUN_NAMES`·`SELECTED_RUNS`에 같은 run을 지정하고 9번만 다시 실행**합니다. 예: v2·10 epoch가 4까지 완료됐다면 동일 v2·10의 5부터 진행합니다. 총 예산을 바꿔 다른 실험으로 이어가지 않습니다. 같은 run을 두 환경에서 동시에 실행하지 않습니다.

In [ ]:
RESTORE_RUN_NAMES = []  # Kaggle 새 세션에서만 예: ["v3_frcnn_e10_s0"]
for name in RESTORE_RUN_NAMES:
    restore_run_from_drive(name)
# 복원 뒤 위의 RESUME_RUN_NAMES와 SELECTED_RUNS에 같은 run을 지정하고 학습 셀만 실행한다.
# 예: RESUME_RUN_NAMES = {"v3_frcnn_e10_s0"}; SELECTED_RUNS = ["v3_frcnn_e10_s0"]


## 11. 공식 val·혼동행렬·AP75–AP95·test 그림 생성

`ANALYZE_RUN_NAMES`에 완료 run을 넣습니다. 동일한 best·데이터의 저장 예측은 재사용합니다. `analysis/val`, `analysis/kaggle_val47`에 두 score의 혼동행렬(건수·행 정규화), TP·FP·FN, micro P/R/F1, 클래스별 표, 오류 이미지를 저장합니다. IoU는 0.75로 고정하며 분모 0은 null/N/A입니다. 혼동행렬은 IoU 우선의 클래스 독립 일대일 매칭으로, 오분류를 FP·FN에 반영합니다.

공식 `val`, `kaggle`, `test_only`는 **모두 val 정답 기반 점수**입니다. `kaggle`은 기존 val 47장이고 리더보드 점수가 아닙니다. `test_only`는 test 전용 18개 클래스를 val에서 평가한 값입니다. 1차 대비 비교는 같은 47장·정답·팀 규칙으로 다시 평가한 결과를 사용합니다.

AP 분석은 score 0.001 예측으로 AP75/80/85/90/95, 평균, AP75−AP95, GT 수를 저장합니다. 정답 없는 클래스는 N/A입니다. 상·하위 곡선은 **평균 AP 상위5·중복 없는 하위5**, GT>0 기준입니다. AP 둘 다 낮아 차이만 작은 경우를 좋은 위치 정확도로 해석하지 않습니다.

`analysis/test`는 test **842장 전체(검출 0 포함)**의 예측 분포, 클래스별 수, 학습·val 빈도와 비교표, score별 검출 수·score 분포, IoU≥0.5 겹침·다른 클래스 겹침, 예시 이미지를 보존합니다. 공통 대회 클래스와 others·비대회 예측을 구분합니다. **Test는 정답이 없으므로 혼동행렬·AP·Precision/Recall/F1을 만들지 않습니다.** 예측 빈도를 실제 정답 분포로 해석하지 않습니다.

In [ ]:
ANALYZE_RUN_NAMES = []  # 예: ["v3_frcnn_e5_s0"]; 학습·추론 캐시 재사용, 그림 추가 생성
for name in ANALYZE_RUN_NAMES:
    match = experiment_plan().loc[lambda frame: frame.run_name.eq(name)]
    if len(match) != 1:
        raise ValueError("9개 실행 계획의 run 이름을 지정하세요.")
    bundle = prepare_bundle(match.iloc[0]["variant"])
    root = RUN_ROOT / name
    if not (root / "result.json").is_file():
        raise RuntimeError("학습 완료 run을 선택하세요. 중단된 run은 먼저 재개하세요.")
    finalize_run(root, bundle, plots=True)


## 12. 선택: 1차 모델을 같은 규칙으로 다시 평가

`FIRST_RUN_DIR`에 1차 best.pt·config.json·class_map.csv가 함께 있는 실제 run 폴더를 지정합니다. 학습하지 않으며 기존 모델의 클래스 역매핑으로 공통 val·test를 예측합니다. 기존 모델 설정이 비교 조건과 다르면 중단합니다. 1차 train 빈도를 확보하지 않은 경우 N/A로 표시합니다. 결과는 `first_round_reevaluation` 아래에 분리합니다. 기존 1차 결과표만으로 공정한 비교가 끝났다고 표시하지 않습니다.

In [ ]:
FIRST_RUN_DIR = None  # best.pt, config.json, class_map.csv가 함께 있는 1차 run 폴더
if FIRST_RUN_DIR is not None:
    reevaluate_first_run(FIRST_RUN_DIR, prepare_bundle("v2"), plots=True)


## 13. 선택: val로 고른 모델의 제출 CSV 생성

`CREATE_SUBMISSION=True`일 때 완료 run 가운데 공식 val 최대(동점은 작은 예산·이름 순)의 best를 선택합니다. 같은 test 캐시에서 대회 클래스·score≥0.01로 팀 공통 제출 함수를 사용합니다. **파일만 만들며 실제 Kaggle 제출은 하지 않습니다.** 리더보드 점수는 실제 제출 후 별도 기록합니다.

In [ ]:
CREATE_SUBMISSION = False
if CREATE_SUBMISSION:
    table = record_run_summary(RUN_ROOT, save=False)
    finished = table.loc[table.state.eq("analysis_complete") & table["val"].notna()]
    if finished.empty:
        raise RuntimeError("val·test 분석을 마친 run이 없습니다. 분석을 먼저 완료하세요.")
    selected = finished.sort_values(["val", "epochs", "run"], ascending=[False, True, True]).iloc[0]
    root = RUN_ROOT / selected.run
    bundle = prepare_bundle(selected.variant)
    predictions, _ = predict_checkpoint(root, bundle, "test")
    count = write_submission([p for p in predictions if p["score"] >= .01], root / "submission.csv",
                             competition_from_classes(bundle["root"] / "classes.csv"))
    write_json(root / "submission_selection.json", {"selected_by": "highest official val; tie lower epoch then name",
        "run": selected.run, "val": float(selected["val"]), "score": .01, "rows": count,
        "actual_kaggle_submission": False})
    sync_run(root, include_wandb=True)
    print("CSV 생성 완료(실제 제출은 하지 않음):", root / "submission.csv")


## 14. 공유 결과와 완료 상태 확인

`runs.csv`에서 9개 run의 미실행·중단·학습 완료·분석 완료를 구분합니다. 각 run에 config·클래스 매핑·데이터 해시·환경 이력·weights(best/last/이전 정상본)·metrics·공식 점수·val/test 예측·manifest·분석 결과·W&B 로그를 보관합니다. Kaggle은 업로드 완료 표와 체크섬으로 전송을 확인합니다.

공유 Drive 사본과 팀 규격 `notebooks/modeling/02-cj-faster-rcnn-exp2/runs.csv`에 요약을 보존합니다. 가중치·데이터·제출 CSV를 Git에 강제로 추가하지 않습니다. 이 노트북 수정 자체는 커밋·push·클라우드 학습을 실행하지 않습니다.

In [ ]:
if RUN_ROOT.exists():
    display(record_run_summary(RUN_ROOT, save=False))
    print("공유 결과 폴더: https://drive.google.com/drive/folders/" + RESULT_DRIVE_FOLDER_ID)
    print("W&B: https://wandb.ai/codeit-prj1-team2/pill-object-detection")
else:
    print("아직 학습 결과가 없습니다. Git 커밋·push, 클라우드 준비부터 함께 진행하세요.")
